In [ ]:
import os
from collections import defaultdict

# ===== Change this to your slices output folder =====
output_dir = r"D:\coronal_slices_filtered_flattened"
# ====================================================

class_counts = defaultdict(int)

for root, dirs, files in os.walk(output_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg')):
            # Class name is the parent folder (AD, CN, MCI)
            class_name = os.path.basename(os.path.dirname(root))
            class_counts[class_name] += 1

print("📊 Slices per class:")
for cls, count in class_counts.items():
    print(f"{cls}: {count}")


In [ ]:
import os
from collections import defaultdict

# ===== Change this to your slices output folder =====
output_dir = r"D:\coronal_slices_filtered_flattened"
# ====================================================

# Dictionary: {split: {class: count}}
split_class_counts = defaultdict(lambda: defaultdict(int))

for split in os.listdir(output_dir):  # train, val, test
    split_path = os.path.join(output_dir, split)
    if not os.path.isdir(split_path):
        continue
    for cls in os.listdir(split_path):  # AD, CN, MCI
        cls_path = os.path.join(split_path, cls)
        if not os.path.isdir(cls_path):
            continue
        count = sum(1 for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg')))
        split_class_counts[split][cls] = count

# Print results
print("📊 Slices per class per split:\n")
for split in split_class_counts:
    print(f"--- {split.upper()} ---")
    for cls, count in split_class_counts[split].items():
        print(f"{cls}: {count}")
    print()


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# ============================
# CONFIGURATION
# ============================
slices_root = r"D:\coronal_slices_filtered_flattened"  # Path to extracted slices
splits = ["train", "val", "test"]
classes = ["AD", "CN", "MCI"]

# ============================
# STEP 1 — Count slices
# ============================
slice_counts = {split: {cls: 0 for cls in classes} for split in splits}

for split in splits:
    for cls in classes:
        folder = os.path.join(slices_root, split, cls)
        count = len([f for f in os.listdir(folder) if f.lower().endswith(".png")])
        slice_counts[split][cls] = count

# ============================
# STEP 2 — Display counts
# ============================
print("📊 Slices per class per split:\n")
total_per_class = {cls: 0 for cls in classes}

for split in splits:
    print(f"--- {split.upper()} ---")
    for cls in classes:
        count = slice_counts[split][cls]
        total_per_class[cls] += count
        print(f"{cls}: {count}")
    print()

print("TOTAL PER CLASS:")
for cls, total in total_per_class.items():
    print(f"{cls}: {total}")

# ============================
# STEP 3 — Plot bar chart
# ============================
x = np.arange(len(classes))
bar_width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, split in enumerate(splits):
    counts = [slice_counts[split][cls] for cls in classes]
    ax.bar(x + i*bar_width, counts, width=bar_width, label=split.capitalize())

ax.set_xlabel("Class")
ax.set_ylabel("Number of Slices")
ax.set_title("Coronal Slices per Class per Split")
ax.set_xticks(x + bar_width)
ax.set_xticklabels(classes)
ax.legend()
plt.tight_layout()
plt.show()

# ============================
# STEP 4 — Compute class weights (for PyTorch)
# ============================
total_slices = sum(total_per_class.values())
class_weights = {cls: total_slices / (len(classes) * total) for cls, total in total_per_class.items()}

print("\n📌 Class Weights for PyTorch:")
for cls, weight in class_weights.items():
    print(f"{cls}: {weight:.4f}")

# As a tensor for PyTorch:
import torch
class_weights_tensor = torch.tensor([class_weights[cls] for cls in classes], dtype=torch.float)
print("\nClass Weights Tensor:", class_weights_tensor)


In [ ]:
import os
import torch

# Path to your coronal slices dataset
dataset_path = r"D:\coronal_slices_filtered_flattened"

# Classes (ensure they are in the same order you'll use for training)
classes = ['AD', 'CN', 'MCI']

slice_counts = {cls: 0 for cls in classes}

# Loop through each split (train, val, test)
for split in ['train', 'val', 'test']:
    for cls in classes:
        cls_dir = os.path.join(dataset_path, split, cls)
        if os.path.exists(cls_dir):
            slice_counts[cls] += len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

print("📊 Slice counts per class:", slice_counts)

# Compute class weights
total_slices = sum(slice_counts.values())
num_classes = len(classes)
class_weights = [total_slices / (num_classes * slice_counts[cls]) for cls in classes]

print("\n📌 Class Weights for PyTorch:")
for cls, w in zip(classes, class_weights):
    print(f"{cls}: {w:.4f}")

weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("\nClass Weights Tensor:", weights_tensor)


In [ ]:
import os
from collections import Counter
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ========= CONFIGURATION =========
data_dir = r"D:\coronal_slices_filtered_flattened"  # root dataset directory
batch_size = 6
image_size = (224, 224)  # resize target for model
num_workers = 4
# =================================

# Normalization values (ImageNet pretrained models)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# ----------- TRANSFORMS -----------
# Train: with augmentation
train_transforms = transforms.Compose([
    transforms.Resize(image_size),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomRotation(15),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Val/Test: no augmentation
val_test_transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ----------- DATASETS -------------
train_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=train_transforms
)
val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "val"),
    transform=val_test_transforms
)
test_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "test"),
    transform=val_test_transforms
)

# ----------- DATALOADERS ----------
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# ----------- CLASS NAMES ----------
print("Classes:", train_dataset.classes)
print(f"Train total: {len(train_dataset)}, Val total: {len(val_dataset)}, Test total: {len(test_dataset)}")

# ----------- COUNT PER CLASS -------
def count_per_class(dataset):
    counts = Counter()
    for _, label in dataset:
        counts[dataset.classes[label]] += 1
    return dict(counts)

print("\n📊 Slices per class per split:")
print("--- TRAIN ---", count_per_class(train_dataset))
print("--- VAL ---", count_per_class(val_dataset))
print("--- TEST ---", count_per_class(test_dataset))

# ----------- CLASS WEIGHTS ---------
all_counts = count_per_class(train_dataset)
total_train = sum(all_counts.values())
class_weights = [total_train / all_counts[cls] for cls in train_dataset.classes]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("\n📌 Class Weights for PyTorch:")
for cls, w in zip(train_dataset.classes, class_weights):
    print(f"{cls}: {w:.4f}")
print("Class Weights Tensor:", class_weights_tensor)


In [ ]:
# now we will keep early convolutional layers frozen (they capture very generic edges/shapes).
# Unfreeze the last ResNet block + the final fc layer for training.
#This gives you the benefit of pretrained knowledge but lets the model specialize
# below is  the complete fine-tuning code for your dataset and class weights:

In [ ]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

# =================== CONFIG ===================
data_dir = r"D:\coronal_slices_filtered_flattened"
batch_size = 8
image_size = (224, 224)
num_workers = 4
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class weights (example)
class_weights = torch.tensor([4.5290, 3.5186, 2.0202], dtype=torch.float).to(device)

# =================== STEP 1: Compute Mean & Std ===================
print("📊 Computing dataset mean and std...")
temp_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor()
])

train_dataset_for_stats = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=temp_transform)
train_loader_for_stats = DataLoader(train_dataset_for_stats, batch_size=batch_size, shuffle=False, num_workers=num_workers)

mean = 0.0
std = 0.0
nb_samples = 0

for data, _ in tqdm(train_loader_for_stats):
    batch_samples = data.size(0)
    data = data.view(batch_samples, data.size(1), -1)  # flatten H*W
    mean += data.mean(2).sum(0)
    std += data.std(2).sum(0)
    nb_samples += batch_samples

mean /= nb_samples
std /= nb_samples

print(f"✅ Computed Mean: {mean}")
print(f"✅ Computed Std: {std}")

# =================== STEP 2: Define Transforms ===================
train_transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])

val_test_transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])

# =================== STEP 3: Load Datasets ===================
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print("Classes:", train_dataset.classes)

# =================== STEP 4: Model ===================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_dataset.classes))
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

# =================== STEP 5: Loss, Optimizer, Scheduler ===================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.8)

# =================== STEP 6: Training Loop ===================
def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, num_epochs=10):
    best_acc = 0.0
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                loader = train_loader
            else:
                model.eval()
                loader = val_loader

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in tqdm(loader):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == 'train':
                scheduler.step()

            # Save best model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), "resedualnetwork50_finetuned_mri.pth")
                print("✅ Best model saved.")

    print(f"\nTraining complete. Best val Acc: {best_acc:.4f}")
    return model

# =================== STEP 7: Train ===================
model = train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, num_epochs)


In [ ]:
#Loads a ResNet50 architecture without pretrained ImageNet weights (weights=None).
#Replaces the final classification layer to match 3 classes (AD, CN, MCI).
#Loads your fine-tuned weights from resnet50_finetuned.pth.
#Removes the final classification layer to make a feature extractor. This means the model outputs high-level features (vector representations) instead of class predictions.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
import pandas as pd
from tqdm import tqdm

# ==========================
# CONFIG
# ==========================
data_dir = r"D:\coronal_slices_filtered_flattened"   # Dataset root
output_csv = "resedualneworkt50_finetuned_features.csv"
batch_size = 6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================
# 1. Load Fine-Tuned Model
# ==========================
model = models.resnet50(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)  # trained with 3 classes

# Load fine-tuned weights (future-safe)
state_dict = torch.load("resedualnetwork50_finetuned_mri.pth", map_location=device, weights_only=True)
model.load_state_dict(state_dict)

# Remove classification head to get features
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor.to(device)
feature_extractor.eval()

# ==========================
# 2. Define Transforms
# ==========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.2772, 0.2772, 0.2772],  # ImageNet normalization
        std=[0.2666, 0.2666, 0.2666]
    )
])

# ==========================
# 3. Load Datasets
# ==========================
datasets_dict = {}
for split in ["train", "val", "test"]:
    datasets_dict[split] = datasets.ImageFolder(os.path.join(data_dir, split), transform=transform)

# ==========================
# 4. Extract Features
# ==========================
all_data = []

with torch.no_grad():
    for split, dataset_split in datasets_dict.items():
        loader = torch.utils.data.DataLoader(dataset_split, batch_size=batch_size, shuffle=False)
        print(f"\nExtracting features for {split} ({len(dataset_split)} images)...")

        for batch_idx, (inputs, labels) in enumerate(tqdm(loader, total=len(loader))):
            inputs = inputs.to(device)
            labels_np = labels.cpu().numpy()

            # Forward pass
            feats = feature_extractor(inputs)
            feats = feats.view(feats.size(0), -1).cpu().numpy()  # flatten

            # Get file paths for current batch
            start_idx = batch_idx * batch_size
            paths = [dataset_split.samples[start_idx + i][0] for i in range(len(labels_np))]
            filenames = [os.path.basename(p) for p in paths]

            # Append to all_data
            for fname, lbl, feat in zip(filenames, labels_np, feats):
                all_data.append([fname, dataset_split.classes[lbl]] + feat.tolist())

# ==========================
# 5. Save to CSV
# ==========================
feature_dim = len(all_data[0]) - 2
columns = ["filename", "class"] + [f"f{i}" for i in range(1, feature_dim + 1)]
df = pd.DataFrame(all_data, columns=columns)
df.to_csv(output_csv, index=False)

print(f"\n✅ Features saved to {output_csv}")


In [ ]:
# explanation of the result above:
#DataLoader splits your dataset into batches of size batch_size (6 in your code).
#Each batch is sent through the model, features are extracted, and stored in all_data.
#The loop continues until all batches have been processed, which means all images in the dataset are included, not just one batch

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

# ==========================
# CONFIG "D:\resedualneworkt50_finetuned_features.csv"
# ==========================
input_csv = r"D:\resedualneworkt50_finetuned_features.csv"
output_csv = "resedualnetwork50_selected_features_cumulative.csv"
correlation_threshold = 0.95
cumulative_importance_threshold = 0.95  # keep features covering 95% of total importance

# ==========================
# 1. Load CSV
# ==========================
df = pd.read_csv(input_csv)
filenames = df['filename']
labels = df['class']
features = df.drop(['filename', 'class'], axis=1)

print(f"Original number of features: {features.shape[1]}")

# ==========================
# 2. Remove low-variance features
# ==========================
var_thresh = VarianceThreshold(threshold=1e-5)
features_var = var_thresh.fit_transform(features)
features_var_cols = features.columns[var_thresh.get_support()]
features_var_df = pd.DataFrame(features_var, columns=features_var_cols)
print(f"After variance threshold: {features_var_df.shape[1]} features")

# ==========================
# 3. Remove highly correlated features
# ==========================
corr_matrix = features_var_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > correlation_threshold)]
features_uncorr_df = features_var_df.drop(columns=to_drop)
print(f"After correlation pruning: {features_uncorr_df.shape[1]} features")

# ==========================
# 4. Random Forest feature importance
# ==========================
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(features_uncorr_df, labels)
importances = rf.feature_importances_

# Sort features by importance
sorted_indices = np.argsort(importances)[::-1]
sorted_importances = importances[sorted_indices]
sorted_features = features_uncorr_df.columns[sorted_indices]

# Cumulative importance
cumulative_importance = np.cumsum(sorted_importances) / np.sum(sorted_importances)
num_features = np.searchsorted(cumulative_importance, cumulative_importance_threshold) + 1
print(f"Number of features selected to cover {cumulative_importance_threshold*100}% importance: {num_features}")

# Select top features based on cumulative importance
selected_features_df = features_uncorr_df[sorted_features[:num_features]]

# ==========================
# 5. Save selected features
# ==========================
final_df = pd.concat([filenames, labels, selected_features_df], axis=1)
final_df.to_csv(output_csv, index=False)
print(f"✅ Selected features saved to {output_csv}")


In [ ]:
import pandas as pd

# Load CSV
csv_path = "resedualnetwork50_selected_features_cumulative.csv"
df = pd.read_csv(csv_path)

# Show basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns[:10], "...")  # first 10 columns
print("\nSample row:\n", df.iloc[0, :10])   # first 10 values of row 0
print("\nData types:\n", df.dtypes.value_counts())


In [ ]:
import pandas as pd
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================
# CONFIG
# ==========================
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
n_neighbors = 30   # UMAP parameter
min_dist = 0.3     # UMAP parameter
random_state = 42

# ==========================
# 1. Load selected features
# ==========================
df = pd.read_csv(input_csv)
labels = df['class']
features = df.drop(['filename', 'class'], axis=1)

print(f"Features shape: {features.shape}")

# ==========================
# 2. UMAP dimensionality reduction
# ==========================
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2, random_state=random_state)
embedding = reducer.fit_transform(features)

# ==========================
# 3. Visualization
# ==========================
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=embedding[:, 0], y=embedding[:, 1],
    hue=labels,
    palette="Set1",
    s=50,
    alpha=0.8
)
plt.title("UMAP Projection of MRI Features (Selected Features)")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Class")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================
# CONFIG
# ==========================
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
n_neighbors = 30       # UMAP
min_dist = 0.3         # UMAP
random_state = 42
n_splits = 5           # Cross-validation folds

# ==========================
# 1. Load data
# ==========================
df = pd.read_csv(input_csv)
X = df.drop(['filename', 'class'], axis=1).values
y = df['class'].values
filenames = df['filename'].values

print(f"Features shape: {X.shape}, Labels shape: {y.shape}")

# ==========================
# 2. Scale features
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================
# 3. Cross-validation training
# ==========================
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
model = RandomForestClassifier(n_estimators=200, random_state=random_state)

y_true_all = []
y_pred_all = []

start_fold = 5  # resume from the next fold
for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), 1):
    if fold < start_fold:
        print(f"Skipping fold {fold}")
        continue

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    
    print(f"Fold {fold} done")

# ==========================
# 4. Confusion Matrix
# ==========================
cm = confusion_matrix(y_true_all, y_pred_all, labels=['AD','CN','MCI'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['AD','CN','MCI'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: Cross-Validated Predictions")
plt.show()

# ==========================
# 5. Classification Report
# ==========================
print("Classification Report:\n")
print(classification_report(y_true_all, y_pred_all, target_names=['AD','CN','MCI']))

# ==========================
# 6. Identify misclassified MCI
# ==========================
results_df = pd.DataFrame({
    'filename': filenames,
    'true': y_true_all,
    'pred': y_pred_all
})

misclassified_mci = results_df[(results_df['true']=='MCI') & (results_df['pred'] != 'MCI')]
print("\nMisclassified MCI samples:")
print(misclassified_mci)

# ==========================
# 7. UMAP visualization
# ==========================
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2, random_state=random_state)
embedding = reducer.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=embedding[:,0], y=embedding[:,1],
    hue=y,
    palette="Set1",
    s=50,
    alpha=0.6
)

# Highlight misclassified MCI samples
mis_idx = [i for i, fname in enumerate(filenames) if fname in misclassified_mci['filename'].values]
plt.scatter(embedding[mis_idx,0], embedding[mis_idx,1],
            facecolors='none', edgecolors='k', s=100, linewidths=1.5,
            label='Misclassified MCI')

plt.title("UMAP Projection of MRI Features with Misclassified MCI Highlighted")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

# ==========================
# CONFIG
# ==========================
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
n_neighbors = 15       # UMAP
min_dist = 0.1         # UMAP
random_state = 42
n_splits = 5           # Cross-validation folds

# ==========================
# 1. Load data
# ==========================
df = pd.read_csv(input_csv)
X = df.drop(['filename', 'class'], axis=1).values
y = df['class'].values
filenames = df['filename'].values

print(f"Features shape: {X.shape}, Labels shape: {y.shape}")

# ==========================
# 2. Encode string labels to integers
# ==========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # 'AD'->0, 'CN'->1, 'MCI'->2
label_names = le.classes_        # ['AD','CN','MCI']

# ==========================
# 3. Scale features
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================
# 4. Cross-validation training with XGBoost
# ==========================
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=random_state)

y_true_all = []
y_pred_all = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_encoded), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    
    print(f"Fold {fold} done")

# Convert numeric labels back to original string labels
y_true_all_str = le.inverse_transform(y_true_all)
y_pred_all_str = le.inverse_transform(y_pred_all)

# ==========================
# 5. Confusion Matrix
# ==========================
cm = confusion_matrix(y_true_all_str, y_pred_all_str, labels=label_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: XGBoost CV Predictions")
plt.show()

# ==========================
# 6. Classification Report
# ==========================
print("Classification Report:\n")
print(classification_report(y_true_all_str, y_pred_all_str, target_names=label_names))

# ==========================
# 7. Identify misclassified samples
# ==========================
results_df = pd.DataFrame({
    'filename': filenames,
    'true': y_true_all_str,
    'pred': y_pred_all_str
})

# Separate misclassified samples by class
misclassified = results_df[results_df['true'] != results_df['pred']]
mis_ad = misclassified[misclassified['true'] == 'AD']
mis_cn = misclassified[misclassified['true'] == 'CN']
mis_mci = misclassified[misclassified['true'] == 'MCI']

print("\nTotal misclassified samples:", len(misclassified))
print("Misclassified AD:", len(mis_ad))
print("Misclassified CN:", len(mis_cn))
print("Misclassified MCI:", len(mis_mci))

# ==========================
# 8. UMAP visualization
# ==========================
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2, random_state=random_state)
embedding = reducer.fit_transform(X_scaled)

plt.figure(figsize=(12, 9))
sns.scatterplot(
    x=embedding[:,0], y=embedding[:,1],
    hue=y_true_all_str,
    palette="Set1",
    s=50,
    alpha=0.6
)

# Highlight misclassified samples
def highlight_samples(subset, color, label):
    idx = [i for i, fname in enumerate(filenames) if fname in subset['filename'].values]
    plt.scatter(embedding[idx,0], embedding[idx,1],
                facecolors='none', edgecolors=color, s=120, linewidths=1.5,
                label=label)

highlight_samples(mis_ad, 'blue', 'Misclassified AD')
highlight_samples(mis_cn, 'green', 'Misclassified CN')
highlight_samples(mis_mci, 'black', 'Misclassified MCI')

plt.title("UMAP Projection of MRI Features with Misclassified Samples Highlighted")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend()
plt.show()


In [ ]:
# explanation of the code below         ( I didnot run this code)
#Tuned XGBoost hyperparameters for better generalization.
#Training vs test accuracy per fold is tracked and plotted.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

# ==========================
# CONFIG 
# ==========================
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
n_neighbors = 15       # UMAP
min_dist = 0.1         # UMAP
random_state = 42
n_splits = 5           # Cross-validation folds

# ==========================
# 1. Load data
# ==========================
df = pd.read_csv(input_csv)
X = df.drop(['filename', 'class'], axis=1).values
y = df['class'].values
filenames = df['filename'].values

print(f"Features shape: {X.shape}, Labels shape: {y.shape}")

# ==========================
# 2. Encode string labels to integers
# ==========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # 'AD'->0, 'CN'->1, 'MCI'->2
label_names = le.classes_        # ['AD','CN','MCI']

# ==========================
# 3. Scale features
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================
# 4. Cross-validation training with XGBoost (tuned)
# ==========================
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=random_state,
    max_depth=4,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2
)

y_true_all = []
y_pred_all = []

train_acc_all = []
test_acc_all = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_encoded), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
    
    model.fit(X_train, y_train)
    
    # Training accuracy
    y_train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, y_train_pred)
    train_acc_all.append(train_acc)
    
    # Test accuracy
    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    test_acc_all.append(test_acc)
    
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    
    print(f"Fold {fold} — Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

# Convert numeric labels back to strings
y_true_all_str = le.inverse_transform(y_true_all)
y_pred_all_str = le.inverse_transform(y_pred_all)

# ==========================
# 5. Confusion Matrix
# ==========================
cm = confusion_matrix(y_true_all_str, y_pred_all_str, labels=label_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: XGBoost CV Predictions")
plt.show()

# ==========================
# 6. Classification Report
# ==========================
print("Classification Report:\n")
print(classification_report(y_true_all_str, y_pred_all_str, target_names=label_names))

# ==========================
# 7. Identify misclassified samples
# ==========================
results_df = pd.DataFrame({
    'filename': filenames,
    'true': y_true_all_str,
    'pred': y_pred_all_str
})

misclassified = results_df[results_df['true'] != results_df['pred']]
mis_ad = misclassified[misclassified['true'] == 'AD']
mis_cn = misclassified[misclassified['true'] == 'CN']
mis_mci = misclassified[misclassified['true'] == 'MCI']

print("\nTotal misclassified samples:", len(misclassified))
print("Misclassified AD:", len(mis_ad))
print("Misclassified CN:", len(mis_cn))
print("Misclassified MCI:", len(mis_mci))

# ==========================
# 8. UMAP visualization
# ==========================
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2, random_state=random_state)
embedding = reducer.fit_transform(X_scaled)

plt.figure(figsize=(12, 9))
sns.scatterplot(
    x=embedding[:,0], y=embedding[:,1],
    hue=y_true_all_str,
    palette="Set1",
    s=50,
    alpha=0.6
)

# Highlight misclassified samples
def highlight_samples(subset, color, label):
    idx = [i for i, fname in enumerate(filenames) if fname in subset['filename'].values]
    plt.scatter(embedding[idx,0], embedding[idx,1],
                facecolors='none', edgecolors=color, s=120, linewidths=1.5,
                label=label)

highlight_samples(mis_ad, 'blue', 'Misclassified AD')
highlight_samples(mis_cn, 'green', 'Misclassified CN')
highlight_samples(mis_mci, 'black', 'Misclassified MCI')

plt.title("UMAP Projection of MRI Features with Misclassified Samples Highlighted")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend()
plt.show()

# ==========================
# 9. Plot training vs test accuracy per fold
# ==========================
folds = np.arange(1, n_splits+1)
plt.plot(folds, train_acc_all, marker='o', label='Train Accuracy')
plt.plot(folds, test_acc_all, marker='s', label='Test Accuracy')
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("Training vs Test Accuracy per Fold")
plt.legend()
plt.show()

print("\nAverage Train Accuracy:", np.mean(train_acc_all))
print("Average Test Accuracy:", np.mean(test_acc_all))


In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# ==========================
# CONFIG
# ==========================
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
random_state = 42
n_splits = 5           # Cross-validation folds

# ==========================
# 1. Load data
# ==========================
df = pd.read_csv(input_csv)
X = df.drop(['filename', 'class'], axis=1).values
y = df['class'].values

print(f"Features shape: {X.shape}, Labels shape: {y.shape}")

# ==========================
# 2. Encode labels
# ==========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # 'AD'->0, 'CN'->1, 'MCI'->2
label_names = le.classes_

# ==========================
# 3. Scale features
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================
# 4. Cross-validation training with XGBoost (tuned)
# ==========================
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=random_state,
    max_depth=4,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2
)

train_acc_all = []
test_acc_all = []

y_true_all = []
y_pred_all = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_encoded), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
    
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Accuracies
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_acc_all.append(train_acc)
    test_acc_all.append(test_acc)
    
    y_true_all.extend(y_test)
    y_pred_all.extend(y_test_pred)
    
    print(f"Fold {fold} — Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
    
    # Confusion Matrices per fold
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    ConfusionMatrixDisplay.from_predictions(
        y_train, y_train_pred,
        display_labels=label_names,
        cmap="Blues", ax=axes[0]
    )
    axes[0].set_title(f"Train Confusion Matrix - Fold {fold}")
    
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_test_pred,
        display_labels=label_names,
        cmap="Blues", ax=axes[1]
    )
    axes[1].set_title(f"Test Confusion Matrix - Fold {fold}")
    
    plt.show()

# ==========================
# 5. Final confusion matrix (all test folds combined)
# ==========================
cm = confusion_matrix(y_true_all, y_pred_all)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Overall Test Confusion Matrix (CV)")
plt.show()

# ==========================
# 6. Classification Report
# ==========================
print("Classification Report (Overall Test):\n")
print(classification_report(y_true_all, y_pred_all, target_names=label_names))

# ==========================
# 7. Accuracy per fold
# ==========================
folds = np.arange(1, n_splits+1)
plt.plot(folds, train_acc_all, marker='o', label='Train Accuracy')
plt.plot(folds, test_acc_all, marker='s', label='Test Accuracy')
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("Training vs Test Accuracy per Fold")
plt.legend()
plt.show()

print("\nAverage Train Accuracy:", np.mean(train_acc_all))
print("Average Test Accuracy:", np.mean(test_acc_all))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.preprocessing import StandardScaler, LabelEncoder
import seaborn as sns

# ===== Config =====
input_csv = "resedualnetwork50_selected_features_cumulative.csv"
n_neighbors = 20
min_dist = 0.0001
random_state = 42

# ===== Load =====
df = pd.read_csv(input_csv)
X = df.drop(['filename', 'class'], axis=1).values
y = df['class'].values

# Encode only for coloring/legend control
le = LabelEncoder()
y_enc = le.fit_transform(y)       # maps to 0..K-1 for plotting
classes = le.classes_             # e.g., ['AD','CN','MCI']

# ===== Scale + UMAP =====
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                    n_components=2, random_state=random_state)
X_umap = reducer.fit_transform(X_scaled)

# ===== Plot =====
plt.figure(figsize=(10, 8))
palette = sns.color_palette("Set1", n_colors=len(classes))
for i, cls in enumerate(classes):
    idx = (y_enc == i)
    plt.scatter(X_umap[idx, 0], X_umap[idx, 1], s=18, alpha=0.75,
                label=cls, color=palette[i])

plt.title("UMAP Clustering of MRI Features (True Classes)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title="Class")
plt.tight_layout()
plt.show()


In [ ]:
# =====================================
# 🧠 ADNI Feature + Metadata Fusion Pipeline
# =====================================

import pandas as pd
import numpy as np
import re
import os

# ============================
# STEP 1: CONFIGURATION
# ============================

# ✅ Update these paths
FEATURES_PATH = r"resedualnetwork50_selected_features_cumulative.csv"        # feature embeddings file (e.g., ResNet features)
METADATA_PATH = r"D:\ADNI1_Merged_MRI_Metadata.csv"       # metadata file (e.g., demographics + diagnosis)

# ============================
# STEP 2: LOAD DATA
# ============================

print("=== Loading datasets ===")
features_df = pd.read_csv(FEATURES_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print(f"Features dataset loaded: {features_df.shape}")
print(f"Metadata dataset loaded: {metadata_df.shape}")

# Display column names for debugging
print("\nFeature columns sample:", features_df.columns[:5].tolist())
print("Metadata columns sample:", metadata_df.columns[:5].tolist())

# ============================
# STEP 3: CLEAN FEATURE FILENAMES AND EXTRACT IMAGE ID
# ============================

def extract_image_id(filename):
    """Extracts IDs like I118678 from filenames."""
    match = re.search(r'I\d+', str(filename))
    return match.group(0) if match else None

if 'filename' not in features_df.columns:
    raise ValueError("❌ The features file must contain a 'filename' column!")

features_df['Image_ID'] = features_df['filename'].apply(extract_image_id)
print("\n✅ Extracted Image_ID from feature filenames:")
print(features_df[['filename', 'Image_ID']].head())

# ============================
# STEP 4: IDENTIFY ID COLUMN IN METADATA
# ============================

metadata_id_col = None
for col in metadata_df.columns:
    # Look for IDs like "I12345"
    if metadata_df[col].astype(str).str.contains(r'^I\d+', regex=True).any():
        metadata_id_col = col
        break

if metadata_id_col is None:
    raise ValueError("❌ No metadata column found that matches pattern Ixxxxxx")

print(f"\n✅ Detected metadata ID column: {metadata_id_col}")

# ============================
# STEP 5: MERGE FEATURES + METADATA
# ============================

print("\n=== Fusing datasets ===")
merged_df = pd.merge(
    features_df,
    metadata_df,
    left_on='Image_ID',
    right_on=metadata_id_col,
    how='inner'
)

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Number of matched records: {len(merged_df)}")

if len(merged_df) == 0:
    print("⚠️ No matches found. Check that Image_ID extraction and metadata ID formats align.")
else:
    print("\n✅ Fusion successful! Preview:")
    display(merged_df.head(5))

# ============================
# STEP 6: CHECK CLASS DISTRIBUTION
# ============================

# Try to find the label column
possible_label_cols = [c for c in merged_df.columns if c.lower() in ['group', 'diagnosis', 'class', 'label']]
if possible_label_cols:
    label_col = possible_label_cols[0]
    print(f"\n✅ Detected label column: {label_col}")
    print(merged_df[label_col].value_counts())
else:
    print("\n⚠️ Could not automatically find a label column (e.g., Group/Diagnosis).")

# ============================
# STEP 7: PREPARE FINAL DATA FOR ANALYSIS
# ============================

# Drop non-numeric columns except labels for model training
non_numeric_cols = merged_df.select_dtypes(exclude=[np.number]).columns.tolist()
keep_cols = ['Image_ID']
if possible_label_cols:
    keep_cols.append(label_col)

merged_df_numeric = merged_df.drop(columns=[c for c in non_numeric_cols if c not in keep_cols], errors='ignore')
print(f"\nFinal numeric dataset shape (for ML): {merged_df_numeric.shape}")

# ============================
# STEP 8: SAVE MERGED OUTPUT
# ============================

OUTPUT_PATH = os.path.join(os.path.dirname(FEATURES_PATH), "ADNI_merged_features_resedual_network_metadata.csv")
merged_df.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Saved fused dataset to: {OUTPUT_PATH}")


In [ ]:
# for fusion using random forest
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# =======================
# Step 1: Load Dataset
# =======================
csv_path = r"ADNI_merged_features_resedual_network_metadata.csv"
df = pd.read_csv(csv_path)

# Separate features and labels
y = df["class"]

# Drop non-numeric columns (like Image_Filename, class)
X = df.drop(columns=["filename", "class"])

# Ensure only numeric data is used
X = X.select_dtypes(include=[np.number])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =======================
# Step 2: Stratified K-Fold CV
# =======================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
all_metrics = []
conf_matrices = []

for train_index, test_index in kf.split(X_scaled, y):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Model (swap with XGBoost or SVM if needed)
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Metrics
    metrics = {
        "Fold": fold,
        "Train Accuracy": accuracy_score(y_train, y_pred_train),
        "Test Accuracy": accuracy_score(y_test, y_pred_test),
        "Precision": precision_score(y_test, y_pred_test, average="weighted"),
        "Recall": recall_score(y_test, y_pred_test, average="weighted"),
        "F1": f1_score(y_test, y_pred_test, average="weighted")
    }
    all_metrics.append(metrics)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_test, labels=np.unique(y))
    conf_matrices.append(cm)

    # Plot per-fold confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Fold {fold} Confusion Matrix")
    plt.show()

    fold += 1

# =======================
# Step 3: Average Results
# =======================
results_df = pd.DataFrame(all_metrics)
print("\nCross-Validation Results:")
print(results_df)

print("\nAverage Performance Across Folds:")
print(results_df.mean(numeric_only=True))

# =======================
# Step 4: Average Confusion Matrix
# =======================
avg_cm = np.mean(conf_matrices, axis=0)

plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title("Average Confusion Matrix Across Folds")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# for fusion using XGBoost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import xgboost as xgb

# ========== Step 1: Load Data ==========
csv_path = r"ADNI_merged_features_resedual_network_metadata.csv"
df = pd.read_csv(csv_path)

# Target label
y = df["Group"]

# Drop unnecessary columns (ID, labels, etc.)
drop_cols = ["filename", "class", "Group","Sex", "Visit","Modality","Description","Type","Acq Date","Format","Downloaded","PHASE","PTID","RID","VISCODE","VISCODE2","VISDATE","ID","SITEID","USERDATE","USERDATE2","update_stamp"]
X = df.drop(columns=drop_cols)

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

# Encode labels (AD, CN, MCI → 0,1,2)
le = LabelEncoder()
y = le.fit_transform(y)

# Standardize numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ========== Step 2: Cross Validation Setup ==========
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store metrics
train_acc, test_acc = [], []
fold = 1

# ========== Step 3: Cross Validation Loop ==========
for train_idx, test_idx in cv.split(X_scaled, y):
    print(f"\n===== Fold {fold} =====")
    
    # Split
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Tuned XGBoost model
    model = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1,
        random_state=42,
        use_label_encoder=False,
        eval_metric="mlogloss"
    )
    
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Accuracies
    train_acc.append(np.mean(y_train_pred == y_train))
    test_acc.append(np.mean(y_test_pred == y_test))
    print("Train Accuracy:", train_acc[-1])
    print("Test Accuracy:", test_acc[-1])
    
    # Classification report for test
    print("\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred, target_names=le.classes_))
    
    # Confusion Matrices (Train + Test side by side)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(confusion_matrix(y_train, y_train_pred), display_labels=le.classes_).plot(ax=axes[0], values_format='d')
    axes[0].set_title(f"Train Confusion Matrix (Fold {fold})")
    
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred), display_labels=le.classes_).plot(ax=axes[1], values_format='d')
    axes[1].set_title(f"Test Confusion Matrix (Fold {fold})")
    
    plt.show()
    
    fold += 1

# ========== Step 4: Overall Performance ==========
print("\n===== Cross-Validation Results =====")
print("Average Train Accuracy:", np.mean(train_acc))
print("Average Test Accuracy:", np.mean(test_acc))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
import xgboost as xgb

# ========== Step 1: Load Data ==========
csv_path = r"ADNI_merged_features_resedual_network_metadata.csv"
df = pd.read_csv(csv_path)

# Target label
y = df["Group"]

# Drop unnecessary columns (ID, labels, etc.)
drop_cols = ["filename", "class", "Group","Sex", "Visit","Modality","Description","Type","Acq Date","Format","Downloaded","PHASE","PTID","RID","VISCODE","VISCODE2","VISDATE","ID","SITEID","USERDATE","USERDATE2","update_stamp"]
X = df.drop(columns=drop_cols)

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

# Encode labels (AD, CN, MCI → 0,1,2)
le = LabelEncoder()
y = le.fit_transform(y)

# Standardize numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ========== Step 2: Cross Validation Setup ==========
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store metrics
train_acc, test_acc = [], []
all_y_true, all_y_pred = [], []
fold = 1

# ========== Step 3: Cross Validation Loop ==========
for train_idx, test_idx in cv.split(X_scaled, y):
    print(f"\n===== Fold {fold} =====")
    
    # Split
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Tuned XGBoost model
    model = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1,
        random_state=42,
        use_label_encoder=False,
        eval_metric="mlogloss"
    )
    
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Collect for aggregation
    all_y_true.extend(y_test)
    all_y_pred.extend(y_test_pred)
    
    # Accuracies
    train_acc.append(np.mean(y_train_pred == y_train))
    test_acc.append(np.mean(y_test_pred == y_test))
    print("Train Accuracy:", train_acc[-1])
    print("Test Accuracy:", test_acc[-1])
    
    # Confusion Matrices (Train + Test per fold)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(confusion_matrix(y_train, y_train_pred), display_labels=le.classes_).plot(ax=axes[0], values_format='d')
    axes[0].set_title(f"Train Confusion Matrix (Fold {fold})")
    
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred), display_labels=le.classes_).plot(ax=axes[1], values_format='d')
    axes[1].set_title(f"Test Confusion Matrix (Fold {fold})")
    
    plt.show()
    
    fold += 1

# ========== Step 4: Aggregated Confusion Matrix ==========
print("\n===== Aggregated Results (All Folds) =====")
cm = confusion_matrix(all_y_true, all_y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Aggregated Confusion Matrix (All Folds)")
plt.show()

# Classification report aggregated
print("\nAggregated Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=le.classes_))

# ========== Step 5: PCA Visualization ==========
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=all_y_pred, cmap="viridis", alpha=0.6)
plt.legend(handles=scatter.legend_elements()[0], labels=le.classes_, title="Predicted")
plt.title("PCA Projection of Samples (Colored by Predicted Class)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis", alpha=0.6)
plt.legend(handles=scatter.legend_elements()[0], labels=le.classes_, title="True")
plt.title("PCA Projection of Samples (Colored by True Class)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [ ]:
# =========================
# ADNI Image Classification and Analysis
# Features + Metadata Fusion (Fixed)
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

# =========================
# CONFIGURATION
# =========================
features_path = r"resedualnetwork50_selected_features_cumulative.csv"
metadata_path = r"D:\ADNI1_Merged_MRI_Metadata.csv"

# =========================
# Step 1: Load Datasets
# =========================
features_df = pd.read_csv(features_path, low_memory=False)
metadata_df = pd.read_csv(metadata_path)

print("Features dataset:", features_df.shape)
print("Metadata dataset:", metadata_df.shape)

# =========================
# Step 2: Prepare merge key
# =========================
# Extract ID pattern like 'I12345' from filename
features_df["Image Data ID"] = features_df["filename"].str.extract(r"(I\d+)")

# =========================
# Step 3: Merge datasets
# =========================
merged_df = pd.merge(features_df, metadata_df, on="Image Data ID", how="inner")
print("✅ Merged dataset shape:", merged_df.shape)

# =========================
# Step 4: Prepare Features
# =========================
# Drop irrelevant identifiers
drop_cols = [
    "filename", "Sex", "Visit", "Modality", "Description", "Type", "Acq Date",
    "Format", "Downloaded", "PHASE", "PTID", "RID", "VISCODE", "VISCODE2",
    "VISDATE", "ID", "SITEID", "USERDATE", "USERDATE2", "update_stamp"
]
for col in drop_cols:
    if col in merged_df.columns:
        merged_df = merged_df.drop(columns=col)

# Determine target label
y = merged_df["class"] if "class" in merged_df.columns else merged_df["Group"]
X = merged_df.drop(columns=[y.name])

print("Feature matrix shape:", X.shape)
print("Labels distribution:\n", y.value_counts())

# Encode categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# =========================
# Step 5: Train/Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Step 6: Handle Missing Values
# =========================
imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# =========================
# Step 7: Scale Features
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# =========================
# Step 8: Train Classifier
# =========================
clf = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)
clf.fit(X_train_scaled, y_train)

# =========================
# Step 9: Evaluate Model
# =========================
y_pred = clf.predict(X_test_scaled)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# =========================
# Step 10: Feature Importance
# =========================
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": clf.feature_importances_
}).sort_values("importance", ascending=False)

sns.barplot(x="importance", y="feature", data=feature_importances.head(20))
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()

# =========================
# Step 11: PCA Visualization
# =========================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

unique_classes = np.unique(y_train)
colors = plt.cm.Set1(np.linspace(0, 1, len(unique_classes)))

plt.figure(figsize=(10, 8))
for i, class_label in enumerate(unique_classes):
    mask = y_train == class_label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[i]], label=class_label, alpha=0.7, s=50)

plt.title("PCA Visualization of Fused Features")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# =========================
# ADNI Image Classification and Analysis
# Features + Metadata Fusion (Fixed)
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

# =========================
# CONFIGURATION
# =========================
features_path = r"resedualnetwork50_selected_features_cumulative.csv"
metadata_path = r"D:\ADNI1_Merged_MRI_Metadata.csv"

# =========================
# Step 1: Load Datasets
# =========================
features_df = pd.read_csv(features_path, low_memory=False)
metadata_df = pd.read_csv(metadata_path)

print("Features dataset:", features_df.shape)
print("Metadata dataset:", metadata_df.shape)

# =========================
# Step 2: Prepare merge key
# =========================
# Extract ID pattern like 'I12345' from filename
features_df["Image Data ID"] = features_df["filename"].str.extract(r"(I\d+)")

# =========================
# Step 3: Merge datasets
# =========================
merged_df = pd.merge(features_df, metadata_df, on="Image Data ID", how="inner")
print("✅ Merged dataset shape:", merged_df.shape)

# =========================
# Step 4: Prepare Features
# =========================
# Drop irrelevant identifiers
drop_cols = [
    "filename", "Sex", "Visit", "Modality", "Description", "Type", "Acq Date",
    "Format", "Downloaded", "PHASE", "PTID", "RID", "VISCODE", "VISCODE2",
    "VISDATE", "ID", "SITEID", "USERDATE", "USERDATE2", "update_stamp", "Subject"
]
for col in drop_cols:
    if col in merged_df.columns:
        merged_df = merged_df.drop(columns=col)

# Determine target label
y = merged_df["class"] if "class" in merged_df.columns else merged_df["Group"]
X = merged_df.drop(columns=[y.name])

print("Feature matrix shape:", X.shape)
print("Labels distribution:\n", y.value_counts())

# Encode categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# =========================
# Step 5: Train/Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Step 6: Handle Missing Values
# =========================
imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# =========================
# Step 7: Scale Features
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# =========================
# Step 8: Train Classifier
# =========================
clf = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)
clf.fit(X_train_scaled, y_train)

# =========================
# Step 9: Evaluate Model
# =========================
y_pred = clf.predict(X_test_scaled)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# =========================
# Step 10: Feature Importance
# =========================
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": clf.feature_importances_
}).sort_values("importance", ascending=False)

sns.barplot(x="importance", y="feature", data=feature_importances.head(20))
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()

# =========================
# Step 11: PCA Visualization
# =========================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

unique_classes = np.unique(y_train)
colors = plt.cm.Set1(np.linspace(0, 1, len(unique_classes)))

plt.figure(figsize=(10, 8))
for i, class_label in enumerate(unique_classes):
    mask = y_train == class_label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[i]], label=class_label, alpha=0.7, s=50)

plt.title("PCA Visualization of Fused Features")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import torch
import clip


In [ ]:
# using clip model
METADATA_PATH = "D:\\ADNI1_Complete 1Yr 1.5T\\ADNI1_Merged_MRI_Metadata.csv"

df = pd.read_csv(METADATA_PATH)
print(df.shape)


In [ ]:
# STEP 2: Identify symptom columns
symptom_cols = [col for col in df.columns if col.startswith("AX")]

print("Number of symptom columns:", len(symptom_cols))
print(symptom_cols)


In [ ]:
# STEP 3: Select required columns

selected_cols = [
    "Image Data ID",
    "Age",
    "Sex",
    "Visit",
    "Subject", "Group"  # label (NOT used in text)
] + symptom_cols

df_demo = df[selected_cols].copy()

# Drop rows with missing essential demographics
df_demo = df_demo.dropna(subset=["Age", "Sex"])

print(df_demo.shape)
df_demo.head()


In [ ]:
# STEP 4: Symptom → Text mapping

symptom_map = {
    "AXNAUSEA": "nausea",
    "AXVOMIT": "vomiting",
    "AXDIARRH": "diarrhea",
    "AXCONSTP": "constipation",
    "AXABDOMN": "abdominal discomfort",
    "AXSWEATN": "excessive sweating",
    "AXDIZZY": "dizziness",
    "AXENERGY": "low energy",
    "AXDROWSY": "drowsiness",
    "AXVISION": "vision problems",
    "AXHDACHE": "headache",
    "AXDRYMTH": "dry mouth",
    "AXBREATH": "shortness of breath",
    "AXCOUGH": "cough",
    "AXPALPIT": "palpitations",
    "AXCHEST": "chest discomfort",
    "AXURNDIS": "urinary discomfort",
    "AXURNFRQ": "frequent urination",
    "AXANKLE": "ankle swelling",
    "AXMUSCLE": "muscle pain",
    "AXRASH": "skin rash",
    "AXINSOMN": "insomnia",
    "AXDPMOOD": "depressed mood",
    "AXCRYING": "frequent crying",
    "AXELMOOD": "emotional lability",
    "AXWANDER": "wandering behavior",
    "AXFALL": "history of falls",
    "AXOTHER": "other reported symptoms"
}


In [ ]:
# STEP 5: Row → Clinical text

def row_to_text(row):
    sex = "male" if row["Sex"] == "M" else "female"
    visit = row["Visit"]

    symptoms = []
    for col, desc in symptom_map.items():
        if row[col] == 1:
            symptoms.append(desc)

    if symptoms:
        symptom_text = "Reported symptoms include " + ", ".join(symptoms) + "."
    else:
        symptom_text = "No significant reported symptoms."

    return (
        f"A {int(row['Age'])}-year-old {sex} subject "
        f"undergoing MRI at visit {visit}. "
        f"{symptom_text}"
    )


In [ ]:
df_demo["text_prompt"] = df_demo.apply(row_to_text, axis=1)

df_demo[["Image Data ID", "text_prompt"]].head()


In [ ]:
# Display all text prompts for all rows
pd.set_option('display.max_colwidth', None)  # Show full text in each cell
print(df_demo[["Image Data ID", "text_prompt"]])

In [ ]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _ = clip.load("ViT-B/32", device=device)
model.eval()


In [ ]:
# Show all text_prompt values without truncation
pd.set_option('display.max_colwidth', None)
display(df_demo[["Image Data ID", "text_prompt"]])

In [ ]:
# If running in a script or some environments, use print instead of display
pd.set_option('display.max_colwidth', None)
print(df_demo[["Image Data ID", "text_prompt"]].to_string(index=False))

In [ ]:
texts = df_demo["text_prompt"].tolist()
text_tokens = clip.tokenize(texts).to(device)

with torch.no_grad():
    clip_embeddings = model.encode_text(text_tokens)

clip_embeddings = clip_embeddings / clip_embeddings.norm(dim=1, keepdim=True)

print("CLIP embedding shape:", clip_embeddings.shape)


In [ ]:
# Install OpenAI CLIP from GitHub
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
# Robustly truncate each prompt so CLIP's tokenizer output is <= 77 tokens
def clip_safe_truncate(text, tokenizer, max_tokens=77):
    words = text.split()
    if not words:
        return ""
    left, right = 1, len(words)
    best = ""
    while left <= right:
        mid = (left + right) // 2
        candidate = ' '.join(words[:mid])
        try:
            tokens = tokenizer([candidate])
            if tokens.shape[1] <= max_tokens:
                best = candidate
                left = mid + 1
            else:
                right = mid - 1
        except RuntimeError:
            right = mid - 1
    # Final check: if even the first word is too long, return empty string
    if best == "":
        return ""
    return best

truncated_texts = [clip_safe_truncate(t, clip.tokenize) for t in df_demo["text_prompt"].tolist()]
text_tokens = clip.tokenize(truncated_texts).to(device)

with torch.no_grad():
    clip_embeddings = model.encode_text(text_tokens)


In [ ]:
# Show the shape of the CLIP embeddings
tprint = print  # alias for clarity
tprint('CLIP embedding shape:', clip_embeddings.shape)

In [ ]:
# Save CLIP embeddings and IDs to a CSV file
import pandas as pd
clip_df = pd.DataFrame(clip_embeddings.cpu().numpy())
clip_df.insert(0, 'Image Data ID', df_demo['Image Data ID'].values)
clip_df.to_csv('clip_embeddings.csv', index=False)
print('CLIP embeddings saved to clip_embeddings.csv')

In [ ]:
# Fuse CLIP embeddings with MRI selected features
import pandas as pd

# Load MRI selected features
mri_features = pd.read_csv('resedualnetwork50_selected_features_cumulative.csv')
# Load CLIP embeddings
clip_df = pd.read_csv('clip_embeddings.csv')

# Merge on Image Data ID
fused_df = pd.merge(mri_features, clip_df, on='Image Data ID', how='inner')

# Save fused features
fused_df.to_csv('fused_mri_clip_features.csv', index=False)
print('Fused features saved to fused_mri_clip_features.csv')

In [ ]:
# Reduce MRI features to 512 using PCA, then fuse with CLIP features
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load MRI selected features and CLIP embeddings
mri_features = pd.read_csv('resedualnetwork50_selected_features_cumulative.csv')
clip_df = pd.read_csv('clip_embeddings.csv')

# Identify MRI feature columns (exclude non-feature columns)
exclude_cols = ['filename', 'class', 'Image Data ID']
mri_feature_cols = [col for col in mri_features.columns if col not in exclude_cols]

# Standardize MRI features
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features[mri_feature_cols])

# Reduce to 512 dimensions
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Create DataFrame for reduced MRI features
mri_pca_df = pd.DataFrame(mri_pca, columns=[f'MRI_PCA_{i+1}' for i in range(512)])
mri_pca_df['Image Data ID'] = mri_features['Image Data ID']

# Merge with CLIP features
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')

# Save fused features
fused_df.to_csv('fused_mri512_clip512_features.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features.csv')

In [ ]:
# Extract Image Data ID from filename for both MRI and CLIP, then fuse after PCA
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import re

# Load MRI selected features and CLIP embeddings
mri_features = pd.read_csv('resedualnetwork50_selected_features_cumulative.csv')
clip_df = pd.read_csv('clip_embeddings.csv')

# Extract Image Data ID from filename in MRI features
if 'Image Data ID' not in mri_features.columns:
    mri_features['Image Data ID'] = mri_features['filename'].apply(lambda x: re.search(r'I\\d+', str(x)).group(0) if re.search(r'I\\d+', str(x)) else np.nan)

# Extract Image Data ID from CLIP if not present
if 'Image Data ID' not in clip_df.columns and 'filename' in clip_df.columns:
    clip_df['Image Data ID'] = clip_df['filename'].apply(lambda x: re.search(r'I\\d+', str(x)).group(0) if re.search(r'I\\d+', str(x)) else np.nan)

# Identify MRI feature columns (float64 only)
feature_cols = mri_features.select_dtypes(include=[np.float64]).columns.tolist()

# Standardize MRI features
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features[feature_cols])

# Reduce to 512 dimensions
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Create DataFrame for reduced MRI features
mri_pca_df = pd.DataFrame(mri_pca, columns=[f'MRI_PCA_{i+1}' for i in range(512)])
mri_pca_df['Image Data ID'] = mri_features['Image Data ID']

# Merge with CLIP features on Image Data ID
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')

# Save fused features
fused_df.to_csv('fused_mri512_clip512_features.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features.csv')

In [ ]:
# Impute missing values in MRI features, then standardize, reduce with PCA, and fuse with CLIP features
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import re

# Load MRI selected features and CLIP embeddings
mri_features = pd.read_csv('resedualnetwork50_selected_features_cumulative.csv')
clip_df = pd.read_csv('clip_embeddings.csv')

# Extract Image Data ID from filename in MRI features
if 'Image Data ID' not in mri_features.columns:
    mri_features['Image Data ID'] = mri_features['filename'].apply(lambda x: re.search(r'I\\d+', str(x)).group(0) if re.search(r'I\\d+', str(x)) else np.nan)

# Extract Image Data ID from CLIP if not present
if 'Image Data ID' not in clip_df.columns and 'filename' in clip_df.columns:
    clip_df['Image Data ID'] = clip_df['filename'].apply(lambda x: re.search(r'I\\d+', str(x)).group(0) if re.search(r'I\\d+', str(x)) else np.nan)

# Identify MRI feature columns (float64 only)
feature_cols = mri_features.select_dtypes(include=[np.float64]).columns.tolist()

# Impute missing values with mean
imputer = SimpleImputer(strategy='mean')
mri_imputed = imputer.fit_transform(mri_features[feature_cols])

# Standardize MRI features
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_imputed)

# Reduce to 512 dimensions
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Create DataFrame for reduced MRI features
mri_pca_df = pd.DataFrame(mri_pca, columns=[f'MRI_PCA_{i+1}' for i in range(512)])
mri_pca_df['Image Data ID'] = mri_features['Image Data ID']

# Merge with CLIP features on Image Data ID
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')

# Save fused features
fused_df.to_csv('fused_mri512_clip512_features.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features.csv')

In [ ]:
# Ensure Image Data ID is string and drop missing before merging
# Drop rows with missing Image Data ID and ensure type is string
mri_pca_df = mri_pca_df.dropna(subset=['Image Data ID'])
clip_df = clip_df.dropna(subset=['Image Data ID'])
mri_pca_df['Image Data ID'] = mri_pca_df['Image Data ID'].astype(str)
clip_df['Image Data ID'] = clip_df['Image Data ID'].astype(str)

# Now merge
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')
fused_df.to_csv('fused_mri512_clip512_features.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features.csv')

In [ ]:
# Debug: Compare Image Data IDs between MRI and CLIP DataFrames
print('MRI PCA DataFrame sample IDs:')
print(mri_pca_df['Image Data ID'].dropna().astype(str).unique()[:10])
print('Total unique MRI IDs:', mri_pca_df['Image Data ID'].nunique())

print('\nCLIP DataFrame sample IDs:')
print(clip_df['Image Data ID'].dropna().astype(str).unique()[:10])
print('Total unique CLIP IDs:', clip_df['Image Data ID'].nunique())

# Intersection count
mri_ids = set(mri_pca_df['Image Data ID'].dropna().astype(str))
clip_ids = set(clip_df['Image Data ID'].dropna().astype(str))
intersection = mri_ids & clip_ids
print('\nNumber of matching IDs:', len(intersection))
if len(intersection) > 0:
    print('Sample matching IDs:', list(intersection)[:10])
else:
    print('No matching IDs found. Check extraction logic and ID formats.')

In [ ]:
# Fix: Correctly extract Image Data ID from MRI filenames and re-merge
import re

# Correct extraction for MRI features
mri_features['Image Data ID'] = mri_features['filename'].apply(lambda x: re.search(r'I\d+', str(x)).group(0) if re.search(r'I\d+', str(x)) else None)

# Redo PCA pipeline if needed (assume mri_pca_df already exists, else rerun PCA steps)
# If you need to rerun PCA, rerun the imputation, scaling, and PCA steps here

# Attach new IDs to PCA DataFrame
mri_pca_df['Image Data ID'] = mri_features['Image Data ID']

# Drop missing and ensure string type
mri_pca_df = mri_pca_df.dropna(subset=['Image Data ID'])
mri_pca_df['Image Data ID'] = mri_pca_df['Image Data ID'].astype(str)
clip_df = clip_df.dropna(subset=['Image Data ID'])
clip_df['Image Data ID'] = clip_df['Image Data ID'].astype(str)

# Merge again
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')
print('Fused shape:', fused_df.shape)
fused_df.to_csv('fused_mri512_clip512_features.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features.csv')

In [ ]:
# Ensure MRI PCA features and IDs are aligned before merging with CLIP
import pandas as pd
import numpy as np
import re
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load MRI features and CLIP embeddings
mri_features = pd.read_csv('resedualnetwork50_selected_features_cumulative.csv')
clip_df = pd.read_csv('clip_embeddings.csv')

# Extract Image Data ID from filename
mri_features['Image Data ID'] = mri_features['filename'].apply(lambda x: re.search(r'I\d+', str(x)).group(0) if re.search(r'I\d+', str(x)) else None)

# Drop rows with missing IDs BEFORE PCA
mri_features = mri_features.dropna(subset=['Image Data ID']).reset_index(drop=True)

# Select only float64 columns (MRI features)
feature_cols = mri_features.select_dtypes(include=[np.float64]).columns.tolist()

# Impute, scale, and PCA
imputer = SimpleImputer(strategy='mean')
mri_imputed = imputer.fit_transform(mri_features[feature_cols])
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_imputed)
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Create DataFrame for reduced MRI features, aligned with IDs
mri_pca_df = pd.DataFrame(mri_pca, columns=[f'MRI_PCA_{i+1}' for i in range(512)])
mri_pca_df['Image Data ID'] = mri_features['Image Data ID'].values

# Prepare CLIP DataFrame
clip_df = clip_df.dropna(subset=['Image Data ID'])
clip_df['Image Data ID'] = clip_df['Image Data ID'].astype(str)
mri_pca_df['Image Data ID'] = mri_pca_df['Image Data ID'].astype(str)

# Merge and save
fused_df = pd.merge(mri_pca_df, clip_df, on='Image Data ID', how='inner')
print('Fused shape:', fused_df.shape)
fused_df.to_csv('fused_mri512_clip512_features_v2.csv', index=False)
print('Fused features (512 MRI + 512 CLIP) saved to fused_mri512_clip512_features_v2.csv')

In [ ]:
import re  # Added to fix NameError

# Get one class label per subject (Image Data ID)
subject_class = mri_df.drop_duplicates(subset=['Image Data ID'])[['Image Data ID', 'class']]

# Merge subject-level class label into each slice in fused_df
fused_df_with_class = pd.merge(fused_df, subject_class, on='Image Data ID', how='left')

# Now fused_df_with_class has a class label for each slice
fused_df_with_class.head()

In [ ]:
# Save the merged DataFrame with class labels to a CSV file
fused_df_with_class.to_csv('fused_slices_with_class_labels.csv', index=False)
print('Saved as fused_slices_with_class_labels.csv')

In [ ]:
# Show the label distribution in the saved CSV file
import pandas as pd

df = pd.read_csv('fused_slices_with_class_labels.csv')
print(df['class'].value_counts())
df['class'].value_counts(normalize=True).plot(kind='bar', title='Class Label Distribution')

In [ ]:
# Inspect DataFrame sizes and join key uniqueness before merging
print('fused_df shape:', fused_df.shape)
print('mri_df shape:', mri_df.shape)
print('Unique Image Data ID in fused_df:', fused_df['Image Data ID'].nunique())
print('Unique Image Data ID in mri_df:', mri_df['Image Data ID'].nunique())
print('Duplicated Image Data ID in mri_df:', mri_df['Image Data ID'].duplicated().sum())
print('Duplicated Image Data ID in fused_df:', fused_df['Image Data ID'].duplicated().sum())

In [ ]:
import pandas as pd

DATA_PATH = r"D:\fused_slices_with_class_labels.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
# ============================================================
# Subject-Level Safe Multimodal Alzheimer's Diagnosis Pipeline
# Using Image Data ID for Grouped Splitting
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

# ------------------------------------------------------------
# 1. Load Fused Multimodal Dataset
# ------------------------------------------------------------
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. Define Columns
# ------------------------------------------------------------
LABEL_COL = "class"
GROUP_COL = "Image Data ID"   # ✅ subject-level grouping key

# Sanity check
assert GROUP_COL in df.columns, "Image Data ID column not found!"
assert LABEL_COL in df.columns, "Class column not found!"

# ------------------------------------------------------------
# 3. Separate Features, Labels, Groups
# ------------------------------------------------------------
X = df.drop(columns=[LABEL_COL, GROUP_COL])
y = df[LABEL_COL]
groups = df[GROUP_COL]

print("Feature shape:", X.shape)
print("Unique subjects:", groups.nunique())
print("Class distribution:\n", y.value_counts())

# ------------------------------------------------------------
# 4. Encode Labels
# ------------------------------------------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Class mapping:")
for c, i in zip(le.classes_, range(len(le.classes_))):
    print(f"{c} -> {i}")

# ------------------------------------------------------------
# 5. SUBJECT-LEVEL Train/Test Split
# ------------------------------------------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y_encoded, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

print("\nTrain subjects:", groups.iloc[train_idx].nunique())
print("Test subjects:", groups.iloc[test_idx].nunique())
print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

# ------------------------------------------------------------
# 6. Train Random Forest Classifier
# ------------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

print("\nTraining Random Forest...")
rf.fit(X_train, y_train)

# ------------------------------------------------------------
# 7. Evaluation
# ------------------------------------------------------------
y_pred = rf.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ------------------------------------------------------------
# 8. Confusion Matrix
# ------------------------------------------------------------
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=le.classes_,
    cmap="Blues",
    xticks_rotation=45
)
plt.title("Subject-Level Confusion Matrix (Multimodal AD Diagnosis)")
plt.show()

# ------------------------------------------------------------
# 9. Save Model & Encoder
# ------------------------------------------------------------
joblib.dump(rf, "multimodal_rf_subject_level.pkl")
joblib.dump(le, "label_encoder.pkl")

print("\nModel and label encoder saved successfully.")


In [ ]:
# ============================================================
# Subject-Level Cross-Validation with XGBoost
# Multimodal AD Diagnosis
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. Define Columns
# ------------------------------------------------------------
LABEL_COL = "class"
GROUP_COL = "Image Data ID"

assert LABEL_COL in df.columns
assert GROUP_COL in df.columns

# ------------------------------------------------------------
# 3. Separate Features, Labels, Groups
# ------------------------------------------------------------
X = df.drop(columns=[LABEL_COL, GROUP_COL])
y = df[LABEL_COL]
groups = df[GROUP_COL]

print("Feature shape:", X.shape)
print("Unique subjects:", groups.nunique())

# ------------------------------------------------------------
# 4. Encode Labels
# ------------------------------------------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(le.classes_)
print("Classes:", le.classes_)

# ------------------------------------------------------------
# 5. GroupKFold Cross-Validation
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=5)

fold_accuracies = []

print("\nStarting Subject-Level Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_encoded, groups), 1):

    print(f"========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    print("Train subjects:", groups.iloc[train_idx].nunique())
    print("Test subjects :", groups.iloc[test_idx].nunique())

    # --------------------------------------------------------
    # 6. XGBoost Model
    # --------------------------------------------------------
    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    # --------------------------------------------------------
    # 7. Train
    # --------------------------------------------------------
    xgb_model.fit(X_train, y_train)

    # --------------------------------------------------------
    # 8. Evaluate
    # --------------------------------------------------------
    y_pred = xgb_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# ------------------------------------------------------------
# 9. Cross-Validation Summary
# ------------------------------------------------------------
print("\n========== Cross-Validation Results ==========")
print("Fold Accuracies:", np.round(fold_accuracies, 4))
print("Mean Accuracy :", np.mean(fold_accuracies))
print("Std Accuracy  :", np.std(fold_accuracies))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Your CV results
fold_accuracies = np.array([0.9515, 0.9693, 0.9594, 0.9679, 0.9568])
mean_acc = fold_accuracies.mean()
std_acc = fold_accuracies.std()

# Plot
plt.figure(figsize=(8,5))
plt.plot(range(1, len(fold_accuracies)+1), fold_accuracies, marker='o', linestyle='-', color='blue', label='Fold Accuracy')
plt.axhline(mean_acc, color='red', linestyle='--', label=f'Mean Accuracy: {mean_acc:.4f}')
plt.fill_between(range(1, len(fold_accuracies)+1), mean_acc-std_acc, mean_acc+std_acc, color='red', alpha=0.1, label=f'Std Dev: {std_acc:.4f}')

plt.xticks(range(1, len(fold_accuracies)+1))
plt.ylim(0.94, 0.975)
plt.xlabel('Fold Number')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Accuracies')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# Subject-Level Cross-Validation with XGBoost
# Confusion Matrix per Fold + Aggregated
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"
df = pd.read_csv(DATA_PATH)

LABEL_COL = "class"
GROUP_COL = "Image Data ID"

# ------------------------------------------------------------
# 2. Prepare Data
# ------------------------------------------------------------
X = df.drop(columns=[LABEL_COL, GROUP_COL])
y = df[LABEL_COL]
groups = df[GROUP_COL]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(le.classes_)
class_names = le.classes_

print("Classes:", class_names)
print("Unique subjects:", groups.nunique())

# ------------------------------------------------------------
# 3. GroupKFold Cross-Validation
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=5)

fold_accuracies = []
cm_aggregate = np.zeros((num_classes, num_classes), dtype=int)

print("\nStarting Subject-Level Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_encoded, groups), 1):

    print(f"========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    print("Train subjects:", groups.iloc[train_idx].nunique())
    print("Test subjects :", groups.iloc[test_idx].nunique())

    # --------------------------------------------------------
    # 4. XGBoost Model
    # --------------------------------------------------------
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    # --------------------------------------------------------
    # 5. Train
    # --------------------------------------------------------
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # 6. Predict
    # --------------------------------------------------------
    y_pred = model.predict(X_test)

    # --------------------------------------------------------
    # 7. Metrics
    # --------------------------------------------------------
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # --------------------------------------------------------
    # 8. Confusion Matrix (Per Fold)
    # --------------------------------------------------------
    cm = confusion_matrix(y_test, y_pred)
    cm_aggregate += cm

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion Matrix – Fold {fold}")
    plt.show()

# ------------------------------------------------------------
# 9. Cross-Validation Summary
# ------------------------------------------------------------
print("\n========== Cross-Validation Summary ==========")
print("Fold Accuracies:", np.round(fold_accuracies, 4))
print("Mean Accuracy :", np.mean(fold_accuracies))
print("Std Accuracy  :", np.std(fold_accuracies))

# ------------------------------------------------------------
# 10. Aggregated Confusion Matrix
# ------------------------------------------------------------
disp_total = ConfusionMatrixDisplay(
    confusion_matrix=cm_aggregate,
    display_labels=class_names
)
disp_total.plot(cmap="Blues", xticks_rotation=45)
plt.title("Aggregated Confusion Matrix (All Folds)")
plt.show()


In [ ]:
# ============================================================
# Subject-Level Binary Classification (AD vs MCI) with XGBoost
# Confusion Matrix + ROC-AUC per Fold + Aggregated
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"
df = pd.read_csv(DATA_PATH)

LABEL_COL = "class"
GROUP_COL = "Image Data ID"

# ------------------------------------------------------------
# 2. Filter for AD vs MCI
# ------------------------------------------------------------
df_binary = df[df[LABEL_COL].isin(["AD", "MCI"])].copy()

le_binary = LabelEncoder()
y_binary = le_binary.fit_transform(df_binary[LABEL_COL])
X_binary = df_binary.drop(columns=[LABEL_COL, GROUP_COL])
groups_binary = df_binary[GROUP_COL]

print("Classes (binary):", le_binary.classes_)
print("Unique subjects:", groups_binary.nunique())

# ------------------------------------------------------------
# 3. GroupKFold Cross-Validation
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=5)

fold_accuracies = []
fold_roc_auc = []
cm_aggregate = np.zeros((2, 2), dtype=int)

print("\nStarting Subject-Level Binary Cross-Validation...\n")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_binary, y_binary, groups_binary), 1):
    print(f"========== Fold {fold} ==========")

    X_train, X_test = X_binary.iloc[train_idx], X_binary.iloc[test_idx]
    y_train, y_test = y_binary[train_idx], y_binary[test_idx]

    print("Train subjects:", groups_binary.iloc[train_idx].nunique())
    print("Test subjects :", groups_binary.iloc[test_idx].nunique())

    # --------------------------------------------------------
    # 4. XGBoost Binary Classifier
    # --------------------------------------------------------
    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    # --------------------------------------------------------
    # 5. Train
    # --------------------------------------------------------
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # 6. Predict
    # --------------------------------------------------------
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1 (MCI)

    # --------------------------------------------------------
    # 7. Metrics
    # --------------------------------------------------------
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)

    roc_auc = roc_auc_score(y_test, y_prob)
    fold_roc_auc.append(roc_auc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Fold {fold} ROC-AUC : {roc_auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le_binary.classes_))

    # --------------------------------------------------------
    # 8. Confusion Matrix (Per Fold)
    # --------------------------------------------------------
    cm = confusion_matrix(y_test, y_pred)
    cm_aggregate += cm

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le_binary.classes_
    )
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion Matrix – Fold {fold}")
    plt.show()

    # --------------------------------------------------------
    # Optional: ROC Curve
    # --------------------------------------------------------
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (All Folds)")
plt.legend()
plt.show()

# ------------------------------------------------------------
# 9. Cross-Validation Summary
# ------------------------------------------------------------
print("\n========== Cross-Validation Summary ==========")
print("Fold Accuracies:", np.round(fold_accuracies, 4))
print("Mean Accuracy :", np.mean(fold_accuracies))
print("Std Accuracy  :", np.std(fold_accuracies))
print("Fold ROC-AUCs :", np.round(fold_roc_auc, 4))
print("Mean ROC-AUC  :", np.mean(fold_roc_auc))

# ------------------------------------------------------------
# 10. Aggregated Confusion Matrix
# ------------------------------------------------------------
disp_total = ConfusionMatrixDisplay(
    confusion_matrix=cm_aggregate,
    display_labels=le_binary.classes_
)
disp_total.plot(cmap="Blues", xticks_rotation=45)
plt.title("Aggregated Confusion Matrix (All Folds)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Your CV results
fold_accuracies = np.array([0.9674, 0.9975, 0.9851, 0.9818, 0.9812])
mean_acc = fold_accuracies.mean()
std_acc = fold_accuracies.std()

# Plot
plt.figure(figsize=(8,5))
plt.plot(range(1, len(fold_accuracies)+1), fold_accuracies, marker='o', linestyle='-', color='blue', label='Fold Accuracy')
plt.axhline(mean_acc, color='red', linestyle='--', label=f'Mean Accuracy: {mean_acc:.4f}')
plt.fill_between(range(1, len(fold_accuracies)+1), mean_acc-std_acc, mean_acc+std_acc, color='red', alpha=0.1, label=f'Std Dev: {std_acc:.4f}')

plt.xticks(range(1, len(fold_accuracies)+1))
plt.ylim(0.94, 0.975)
plt.xlabel('Fold Number')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Accuracies')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"

TARGET_COL = "class"
GROUP_COL = "Image Data ID"

df = pd.read_csv(DATA_PATH)

# Encode labels
le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df[TARGET_COL])

X = df.drop(columns=[TARGET_COL, GROUP_COL])
y = df[TARGET_COL]
groups = df[GROUP_COL]

feature_names = X.columns


In [ ]:
gkf = GroupKFold(n_splits=5)

all_true, all_pred = [], []
all_shap_values = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # ---- XGBoost Model ----
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(le.classes_),
        eval_metric="mlogloss",
        random_state=42,
        tree_method="hist"
    )

    model.fit(X_train, y_train)

    # ---- Evaluation ----
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    all_true.extend(y_test)
    all_pred.extend(preds)

    # ---- Confusion Matrix ----
    cm = confusion_matrix(y_test, preds)

    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_,
                yticklabels=le.classes_)
    plt.title(f"Fold {fold} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # ---- SHAP ----
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    all_shap_values.append(shap_values)

    # ---- SHAP Summary (Fold-wise) ----
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        show=True
    )


In [ ]:
cm_all = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(5, 5))
sns.heatmap(cm_all, annot=True, fmt="d", cmap="Greens",
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title("Aggregated Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
COMPLETE CODE: XGBoost + SHAP (MRI vs Clinical Contribution)


In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# =========================
# 2. LOAD FUSED DATASET
# =========================
DATA_PATH = r"D:\fused_slices_with_class_labels.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


In [ ]:
# =========================
# 3. PREPARE FEATURES & LABELS
# =========================
X = df.drop(columns=["class"])
y_raw = df["class"]

# Encode class labels
le = LabelEncoder()
y = le.fit_transform(y_raw)

class_names = le.classes_
print("Classes:", class_names)

feature_names = X.columns.tolist()


In [ ]:
# =========================
# 4. DEFINE MODALITY GROUPS
# =========================
# MRI features (PCA + CNN features)
mri_features = [c for c in feature_names if c.startswith("MRI_") or c.isdigit()]

# Clinical embeddings (demographics + symptoms)
clinical_features = [c for c in feature_names if c not in mri_features]

print("MRI features:", len(mri_features))
print("Clinical features:", len(clinical_features))


In [ ]:
# All feature columns (exclude label)
feature_cols = df.drop(columns=["class"]).columns.tolist()

# Number of CLIP clinical embeddings
CLIP_DIM = 512

# Clinical features = LAST 512 columns
clinical_features = feature_cols[-CLIP_DIM:]

# MRI features = EVERYTHING ELSE
mri_features = feature_cols[:-CLIP_DIM]

print("MRI features:", len(mri_features))
print("Clinical features:", len(clinical_features))


In [ ]:
# =========================
# 5. TRAIN XGBOOST MODEL
# =========================
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(class_names),
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X, y)


In [ ]:
# =========================
# 6. COMPUTE SHAP VALUES
# =========================
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Select AD class for explanation
ad_index = list(class_names).index("AD")
shap_ad = shap_values[ad_index]


In [ ]:
# =========================
# 7. SHAP SUMMARY (ALL FEATURES)
# =========================
shap.summary_plot(
    shap_ad,
    X,
    show=True
)


In [ ]:
# =========================
# 8. MRI-ONLY SHAP SUMMARY
# =========================
mri_idx = [feature_names.index(f) for f in mri_features]

shap.summary_plot(
    shap_ad[:, mri_idx],
    X[mri_features],
    title="MRI Feature Contribution (SHAP)",
    show=True
)


In [ ]:
# =========================
# 9. CLINICAL-ONLY SHAP SUMMARY
# =========================
clinical_idx = [feature_names.index(f) for f in clinical_features]

shap.summary_plot(
    shap_ad[:, clinical_idx],
    X[clinical_features],
    title="Clinical Feature Contribution (Demographics + Symptoms)",
    show=True
)


In [ ]:
# =========================
# 10. MODALITY-LEVEL CONTRIBUTION
# =========================
mri_contribution = np.mean(np.abs(shap_ad[:, mri_idx]))
clinical_contribution = np.mean(np.abs(shap_ad[:, clinical_idx]))

plt.figure(figsize=(6, 4))
plt.bar(
    ["MRI Features", "Clinical Features"],
    [mri_contribution, clinical_contribution]
)
plt.ylabel("Mean |SHAP Value|")
plt.title("Modality-wise Contribution to AD Prediction")
plt.show()


In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import cv2
from tqdm import tqdm
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# ==========================
# CONFIG
# ==========================
data_dir = r"D:\coronal_slices_filtered_flattened\test"   # MUST contain AD/CN/MCI
model_path = r"D:\mri_classification_model.pth"

output_mis = "misclassified_AD_gradcam"
output_correct = "correct_AD_gradcam"

batch_size = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(os.path.join(output_mis, "side_by_side"), exist_ok=True)
os.makedirs(os.path.join(output_correct, "side_by_side"), exist_ok=True)

# ==========================
# MODEL DEFINITION (MATCHES CHECKPOINT)
# ==========================
class MRIClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

# ==========================
# LOAD MODEL
# ==========================
model = MRIClassifier(num_classes=3)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

# ==========================
# DATA LOADER
# ==========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(data_dir, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

# ==========================
# GRAD-CAM++ SETUP
# ==========================
target_layers = [model.backbone.layer4[-1]]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

total_focus_ratio = []
correct = 0
misclassified = 0

# ==========================
# GRAD-CAM LOOP
# ==========================
for i, (img, label) in enumerate(tqdm(loader, desc="Generating Grad-CAM++")):
    img = img.to(device)
    label = label.to(device)

    img.requires_grad_()  # IMPORTANT

    outputs = model(img)
    preds = outputs.argmax(dim=1)

    # --------------------------
    # Recover RGB image
    # --------------------------
    rgb_img = img[0].permute(1, 2, 0).detach().cpu().numpy()
    rgb_img = np.clip(
        rgb_img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406],
        0, 1
    )

    # --------------------------
    # Grad-CAM++
    # --------------------------
    grayscale_cam = cam(input_tensor=img)[0]

    # --------------------------
    # Hippocampal ROI (Approx.)
    # --------------------------
    h, w = grayscale_cam.shape
    hippocampus_mask = np.zeros((h, w), dtype=np.float32)

    x1, x2 = int(w * 0.35), int(w * 0.65)
    y1, y2 = int(h * 0.55), int(h * 0.85)
    hippocampus_mask[y1:y2, x1:x2] = 1.0

    masked_cam = grayscale_cam * hippocampus_mask
    masked_cam /= (masked_cam.max() + 1e-8)

    focus_ratio = masked_cam.sum() / (grayscale_cam.sum() + 1e-8)
    total_focus_ratio.append(focus_ratio)

    # --------------------------
    # Visualization
    # --------------------------
    vis_full = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    vis_roi = show_cam_on_image(rgb_img, masked_cam, use_rgb=True)

    rgb_cv = (rgb_img * 255).astype(np.uint8)[:, :, ::-1]
    vis_full = vis_full[:, :, ::-1]
    vis_roi = vis_roi[:, :, ::-1]

    side_by_side = np.hstack([rgb_cv, vis_full, vis_roi])

    fname = os.path.basename(dataset.samples[i][0])
    save_dir = output_correct if preds == label else output_mis

    save_path = os.path.join(
        save_dir,
        "side_by_side",
        f"{fname}_pred{idx_to_class[preds.item()]}_true{idx_to_class[label.item()]}.jpg"
    )

    cv2.imwrite(save_path, side_by_side)

    if preds == label:
        correct += 1
    else:
        misclassified += 1

    print(f"{fname} → Hippocampal focus: {focus_ratio:.2%}")

# ==========================
# SUMMARY
# ==========================
print("\n================ SUMMARY ================")
print(f"Correct       : {correct}")
print(f"Misclassified : {misclassified}")
print(f"Avg Hippo Focus Ratio: {np.mean(total_focus_ratio):.2%}")
print(f"Saved to:\n{output_correct}/side_by_side\n{output_mis}/side_by_side")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        # lstm_out = (B, T, H)
        weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(weights * lstm_out, dim=1)
        return context, weights


In [ ]:
class LSTMAttentionModel(nn.Module):
    def __init__(self, seq_len=4, input_dim=256, hidden_dim=128, num_classes=3):
        super().__init__()
        self.seq_len = seq_len
        self.input_dim = input_dim

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=False
        )

        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # reshape input into (B, seq_len, input_dim)
        x = x.view(-1, self.seq_len, self.input_dim)

        lstm_out, _ = self.lstm(x)
        context, attn_weights = self.attention(lstm_out)

        logits = self.fc(context)
        return logits, attn_weights



In [ ]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
df = pd.read_csv(r"D:\fused_mri512_clip512_features.csv")

# Last column is the class label
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["label"])

y = df["label"].values
X = df.drop(columns=["label"]).values

# Check correct feature length
print("Feature length:", X.shape[1])


In [ ]:
import pandas as pd

df = pd.read_csv(r"D:\fused_mri512_clip512_features.csv")
print(df.columns)
print("Number of columns:", len(df.columns))


In [ ]:
import pandas as pd

df = pd.read_csv("D:/fused_slices_with_class_labels.csv")
print(df.columns)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("D:/fused_slices_with_class_labels.csv")

print(df.shape)
print(df.columns[-10:])  # confirm last columns include "class"

# Separate features and label
X = df.drop(columns=["class"]).values
y = LabelEncoder().fit_transform(df["class"])
class_names = np.unique(df["class"])

# Convert to float32 for PyTorch
X = X.astype(np.float32)

# =========================================================
# 2. RESHAPE FOR LSTM
# LSTM expects: (batch, sequence_length, features_per_step)
# We convert 1025 → sequence of length 25 with 41 features each
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN  # 1025 // 25 = 41

X = X[:, :SEQ_LEN * FEAT_PER_STEP]            # trim clean
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)

print("LSTM Input Shape:", X_seq.shape)  # (100608, 25, 41)

# =========================================================
# 3. LSTM + ATTENTION MODEL
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):  # lstm_output = (batch, seq_len, hidden_dim)
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights


class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        context, weights = self.attn(lstm_output)
        out = self.fc(context)
        return out, weights


# Hyperparameters
HIDDEN = 64
NUM_CLASSES = len(class_names)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 4. TRAINING FUNCTION
# =========================================================
def train_model(model, criterion, optimizer, X_train, y_train, epochs=5):
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        inputs = torch.tensor(X_train).to(device)
        labels = torch.tensor(y_train).to(device)

        outputs, _ = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


# =========================================================
# 5. EVALUATION FUNCTION
# =========================================================
def evaluate_model(model, X_test):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_test).to(device)
        outputs, _ = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
    return preds

# =========================================================
# 6. STRATIFIED K-FOLD CROSS VALIDATION
# =========================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1

all_conf_matrices = []

for train_idx, test_idx in skf.split(X_seq, y):
    print(f"\n================ Fold {fold} ================")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")

    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()

    train_model(model, criterion, optimizer, X_train, y_train, epochs=5)

    preds = evaluate_model(model, X_test)

    # Confusion matrix for this fold
    cm = confusion_matrix(y_test, preds)
    all_conf_matrices.append(cm)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()

    fold += 1

# =========================================================
# 7. AGGREGATED CONFUSION MATRIX
# =========================================================
sum_cm = np.sum(all_conf_matrices, axis=0)

plt.figure(figsize=(6, 5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load your fused file
df = pd.read_csv("D:/fused_slices_with_class_labels.csv")

print("Original shape:", df.shape)

# -------------------------------
# 1. Remove all non-numeric columns except the class label
# -------------------------------
label_column = "class"

# Identify non-numeric columns
non_numeric_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove non-numeric columns except "class"
cols_to_drop = [col for col in non_numeric_cols if col != label_column]

print("Dropping these non-numeric columns:", cols_to_drop)

df = df.drop(columns=cols_to_drop)

print("After dropping non-numeric:", df.shape)

# -------------------------------
# 2. Extract features + label
# -------------------------------
X = df.drop(columns=[label_column]).values
y = LabelEncoder().fit_transform(df[label_column])

print("Feature shape:", X.shape)
print("Classes:", np.unique(df[label_column]))

# -------------------------------
# 3. Convert to float32
# -------------------------------
X = X.astype(np.float32)



In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv(r"D:/fused_slices_with_class_labels.csv")
print("Original shape:", df.shape)

label_column = "class"

# Drop all non-numeric columns except the class label
non_numeric_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [col for col in non_numeric_cols if col != label_column]
print("Dropping these non-numeric columns:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

print("After dropping non-numeric:", df.shape)

# Extract features + label
X = df.drop(columns=[label_column]).values
y = LabelEncoder().fit_transform(df[label_column])
class_names = np.unique(df[label_column])
print("Feature shape:", X.shape)
print("Classes:", class_names)

# Convert to float32
X = X.astype(np.float32)

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]  # trim to clean multiple
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM Input Shape:", X_seq.shape)  # (num_samples, seq_len, features)

# =========================================================
# 3. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):  # lstm_output = (batch, seq_len, hidden_dim)
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        context, weights = self.attn(lstm_output)
        out = self.fc(context)
        return out, weights

# =========================================================
# 4. TRAINING & EVALUATION FUNCTIONS (MINI-BATCH)
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, X_train, y_train, epochs=5, batch_size=32):
    model.train()
    
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.long)
    
    train_dataset = TensorDataset(X_tensor, y_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()  # free memory

def evaluate_model(model, X_test, batch_size=64):
    model.eval()
    preds_list = []
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_dataset = TensorDataset(X_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    with torch.no_grad():
        for inputs in test_loader:
            inputs = inputs[0].to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
    
    return np.concatenate(preds_list)

# =========================================================
# 5. STRATIFIED K-FOLD CROSS-VALIDATION
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_conf_matrices = []

for train_idx, test_idx in skf.split(X_seq, y):
    print(f"\n================ Fold {fold} ================")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()
    
    # Train
    train_model(model, criterion, optimizer, X_train, y_train, epochs=10, batch_size=32)
    
    # Evaluate
    preds = evaluate_model(model, X_test, batch_size=64)
    
    # Confusion matrix for this fold
    cm = confusion_matrix(y_test, preds)
    all_conf_matrices.append(cm)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 6. AGGREGATED CONFUSION MATRIX
# =========================================================
sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6, 5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA
# =========================================================
csv_path = r"D:/fused_slices_with_class_labels.csv"
chunksize = 10000
label_column = "class"

X_list = []
y_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col != label_column]
    chunk = chunk.drop(columns=cols_to_drop)
    
    X_chunk = chunk.drop(columns=[label_column]).values.astype('float32')
    y_chunk = chunk[label_column].values
    X_list.append(X_chunk)
    y_list.append(y_chunk)

X = np.vstack(X_list)
y_raw = np.concatenate(y_list)

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

print("Final feature shape:", X.shape)
print("Number of classes:", len(class_names))

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 3. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 4. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        out = self.fc(context)
        return out, weights

# =========================================================
# 5. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 6. STRATIFIED K-FOLD TRAINING WITH CLASSIFICATION REPORT
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in skf.split(X_seq, y):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    # Confusion matrix
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 7. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA
# =========================================================
csv_path = r"D:/fused_slices_with_class_labels.csv"
chunksize = 10000
label_column = "class"
patient_column = "Image Data ID"  # <-- replace with your patient ID column in CSV

X_list = []
y_list = []
patient_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    # Drop non-numeric columns except label and patient ID
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col not in [label_column, patient_column]]
    chunk = chunk.drop(columns=cols_to_drop)
    
    X_chunk = chunk.drop(columns=[label_column, patient_column]).values.astype('float32')
    y_chunk = chunk[label_column].values
    p_chunk = chunk[patient_column].values
    
    X_list.append(X_chunk)
    y_list.append(y_chunk)
    patient_list.append(p_chunk)

# Concatenate all chunks
X = np.vstack(X_list)
y_raw = np.concatenate(y_list)
patient_ids = np.concatenate(patient_list)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

print("Final feature shape:", X.shape)
print("Number of classes:", len(class_names))
print("Number of unique patients:", len(np.unique(patient_ids)))

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 3. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 4. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, weights

# =========================================================
# 5. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 6. GROUP K-FOLD TRAINING
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

gkf = GroupKFold(n_splits=5)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in gkf.split(X_seq, y, groups=patient_ids):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    # Confusion matrix
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 7. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# NOW we will do binary classification

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. LOAD CSV IN CHUNKS (MEMORY SAFE)
# =========================================================
chunks = pd.read_csv("D:/fused_slices_with_class_labels.csv", chunksize=50000)
df = pd.concat(chunks, ignore_index=True)
print("Loaded:", df.shape)

# =========================================================
# 2. KEEP ONLY AD + MCI
# =========================================================
df = df[df["class"].isin(["AD", "MCI"])].reset_index(drop=True)
print("After filtering AD/MCI:", df.shape)

# =========================================================
# 3. BINARY LABEL ENCODING
# =========================================================
df["class"] = df["class"].map({"MCI": 0, "AD": 1})
y = df["class"].values

# =========================================================
# 4. EXTRACT PATIENT ID (change column name if needed)
# =========================================================
patient_ids = df["Image Data ID"].values       # <--- CHANGE if needed

# =========================================================
# 5. DROP NON-NUMERIC FEATURES
# =========================================================
non_numeric = df.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [c for c in non_numeric if c not in ["class"]]
df = df.drop(columns=cols_to_drop)

# Features only
X = df.drop(columns=["class"]).values.astype(np.float32)

# =========================================================
# 6. RESHAPE INTO SEQUENCES FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)

# =========================================================
# 7. LSTM + ATTENTION MODEL
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        context, weights = self.attn(lstm_output)
        out = self.fc(context)
        return out, weights

# =========================================================
# 8. DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 9. GROUP K-FOLD (PATIENT LEVEL SPLIT)
# =========================================================
gkf = GroupKFold(n_splits=5)
fold = 1

for train_idx, test_idx in gkf.split(X_seq, y, groups=patient_ids):
    print(f"\n============= FOLD {fold} =============")

    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_train), torch.tensor(y_train)),
        batch_size=64, shuffle=True)

    test_loader = DataLoader(TensorDataset(
        torch.tensor(X_test), torch.tensor(y_test)),
        batch_size=64, shuffle=False)

    model = LSTM_Attention(FEAT_PER_STEP, 64, 2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # ---------------------- TRAINING ----------------------
    model.train()
    for epoch in range(5):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, _ = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/5 - Loss: {loss.item():.4f}")

    # ---------------------- EVALUATION ----------------------
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            out, _ = model(xb)
            preds.extend(out.argmax(dim=1).cpu().numpy())

    print("\nAccuracy:", accuracy_score(y_test, preds))
    print("\nClassification Report\n", classification_report(y_test, preds))

    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["MCI", "AD"], yticklabels=["MCI", "AD"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()

    fold += 1


In [ ]:
print(df.columns)


In [ ]:
import pandas as pd

meta = pd.read_csv("D:/ADNI1_Merged_MRI_Metadata.csv")
print(meta.columns)


In [ ]:
import pandas as pd

# Load your fused features
df = pd.read_csv("D:/fused_slices_with_class_labels.csv")

# Load original metadata
meta = pd.read_csv("D:/ADNI1_Merged_MRI_Metadata.csv")

# Keep only Image Data ID + PTID
meta_small = meta[["Image Data ID", "PTID"]]

# Merge PTID back into fused CSV
df_merged = df.merge(meta_small, on="Image Data ID", how="left")

print(df_merged.shape)
print(df_merged[["Image Data ID", "PTID"]].head())

# Save new CSV
df_merged.to_csv("D:/fused_with_patient_id.csv", index=False)

print("New CSV saved with PTID included.")


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. LOAD MERGED CSV (WITH PTID)
# =========================================================
df = pd.read_csv(r"D:/fused_with_patient_id.csv")
print("Loaded:", df.shape)

# =========================================================
# 2. KEEP ONLY AD + MCI
# =========================================================
df = df[df["class"].isin(["AD", "MCI"])].reset_index(drop=True)
print("After filtering AD/MCI:", df.shape)

# =========================================================
# 3. Binary encode labels
# =========================================================
df["class"] = df["class"].map({"MCI": 0, "AD": 1})
y = df["class"].values

# =========================================================
# 4. Extract patient IDs for GroupKFold
# =========================================================
groups = df["PTID"].values   # THE FIX — prevents cheating

# =========================================================
# 5. Drop non-numeri columns
# =========================================================
drop_cols = ["Image Data ID", "PTID"]
X = df.drop(columns=drop_cols + ["class"]).values.astype(np.float32)

print("Final feature shape:", X.shape)
# =========================================================
# 6. Reshape into LSTM sequences
# =========================================================
SEQ_LEN = 25
feat_per_step = X.shape[1] // SEQ_LEN

X = X[:, :SEQ_LEN * feat_per_step]
X_seq = X.reshape(len(X), SEQ_LEN, feat_per_step)

print("LSTM input shape:", X_seq.shape)

# =========================================================
# 7. Define LSTM + Attention model
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights


class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, attn_weights = self.attn(lstm_out)
        out = self.fc(context)
        return out, attn_weights


# =========================================================
# 8. Training settings
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

gkf = GroupKFold(n_splits=5)
fold = 1

# =========================================================
# 9. FOLD TRAINING
# =========================================================
for train_idx, test_idx in gkf.split(X_seq, y, groups):

    print(f"\n================= FOLD {fold} =================")

    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
        batch_size=128,
        shuffle=False
    )

    model = LSTM_Attention(input_dim=feat_per_step, hidden_dim=64, num_classes=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # -------- TRAIN --------
    model.train()
    for epoch in range(7):
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, _ = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/7 - Loss: {total_loss:.4f}")

    # -------- EVALUATE --------
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            out,_ = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())

    acc = accuracy_score(y_test, preds)
    print("\nAccuracy:", acc)
    print("\nClassification Report:\n", classification_report(y_test, preds))

    # Confusion matrix
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["MCI", "AD"], yticklabels=["MCI", "AD"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()

    fold += 1


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("D:/fused_with_patient_id.csv")
print("Original shape:", df.shape)

label_column = "class"

# Drop all non-numeric columns except the class label
non_numeric_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [col for col in non_numeric_cols if col != label_column]
print("Dropping these non-numeric columns:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

print("After dropping non-numeric:", df.shape)

# Extract features + label
X = df.drop(columns=[label_column]).values
y = LabelEncoder().fit_transform(df[label_column])
class_names = np.unique(df[label_column])
print("Feature shape:", X.shape)
print("Classes:", class_names)

# Convert to float32
X = X.astype(np.float32)

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]  # trim to clean multiple
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM Input Shape:", X_seq.shape)  # (num_samples, seq_len, features)

# =========================================================
# 3. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):  # lstm_output = (batch, seq_len, hidden_dim)
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        context, weights = self.attn(lstm_output)
        out = self.fc(context)
        return out, weights

# =========================================================
# 4. TRAINING & EVALUATION FUNCTIONS (MINI-BATCH)
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, X_train, y_train, epochs=5, batch_size=32):
    model.train()
    
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.long)
    
    train_dataset = TensorDataset(X_tensor, y_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()  # free memory

def evaluate_model(model, X_test, batch_size=64):
    model.eval()
    preds_list = []
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_dataset = TensorDataset(X_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    with torch.no_grad():
        for inputs in test_loader:
            inputs = inputs[0].to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
    
    return np.concatenate(preds_list)

# =========================================================
# 5. STRATIFIED K-FOLD CROSS-VALIDATION
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_conf_matrices = []

for train_idx, test_idx in skf.split(X_seq, y):
    print(f"\n================ Fold {fold} ================")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()
    
    # Train
    train_model(model, criterion, optimizer, X_train, y_train, epochs=10, batch_size=32)
    
    # Evaluate
    preds = evaluate_model(model, X_test, batch_size=64)
    
    # Confusion matrix for this fold
    cm = confusion_matrix(y_test, preds)
    all_conf_matrices.append(cm)
    
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 6. AGGREGATED CONFUSION MATRIX
# =========================================================
sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6, 5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA
# =========================================================
csv_path = r"D:/fused_with_patient_id.csv"
chunksize = 10000
label_column = "class"
patient_column = "Image Data ID"  # <-- replace with your patient ID column in CSV

X_list = []
y_list = []
patient_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    # Drop non-numeric columns except label and patient ID
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col not in [label_column, patient_column]]
    chunk = chunk.drop(columns=cols_to_drop)
    
    X_chunk = chunk.drop(columns=[label_column, patient_column]).values.astype('float32')
    y_chunk = chunk[label_column].values
    p_chunk = chunk[patient_column].values
    
    X_list.append(X_chunk)
    y_list.append(y_chunk)
    patient_list.append(p_chunk)

# Concatenate all chunks
X = np.vstack(X_list)
y_raw = np.concatenate(y_list)
patient_ids = np.concatenate(patient_list)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

print("Final feature shape:", X.shape)
print("Number of classes:", len(class_names))
print("Number of unique patients:", len(np.unique(patient_ids)))

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 3. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 4. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, weights

# =========================================================
# 5. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 6. GROUP K-FOLD TRAINING
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

gkf = GroupKFold(n_splits=5)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in gkf.split(X_seq, y, groups=patient_ids):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    criterion = nn.CrossEntropyLoss()
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    # Confusion matrix
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 7. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA (AD vs MCI)
# =========================================================
csv_path = r"D:/fused_with_patient_id.csv"
chunksize = 10000
label_column = "class"
patient_column = "Image Data ID"

X_list = []
y_list = []
patient_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    # Drop non-numeric columns except label and patient ID
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col not in [label_column, patient_column]]
    chunk = chunk.drop(columns=cols_to_drop)
    
    # Extract features (drop label and patient ID)
    X_chunk = chunk.drop(columns=[label_column, patient_column]).values.astype('float32')
    
    # Extract labels and patient IDs
    y_chunk = chunk[label_column].values
    p_chunk = chunk[patient_column].values
    
    # Append to lists
    X_list.append(X_chunk)
    y_list.append(y_chunk)
    patient_list.append(p_chunk)

# Concatenate all chunks
X = np.vstack(X_list)
y_raw = np.concatenate(y_list)
patient_ids = np.concatenate(patient_list)

# ==============================
# FILTER TO AD AND MCI ONLY
# ==============================
mask = np.isin(y_raw, ["AD", "MCI"])
X = X[mask]
y_raw = y_raw[mask]
patient_ids = patient_ids[mask]

# Encode labels: AD=0, MCI=1
y = np.array([0 if label=="AD" else 1 for label in y_raw])
class_names = ["AD", "MCI"]

print("Filtered feature shape:", X.shape)
print("Number of classes:", len(class_names))
print("Number of unique patients:", len(np.unique(patient_ids)))

# =========================================================
# 2. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]  # trim excess if not divisible
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 3. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 4. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, weights

# =========================================================
# 5. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 6. GROUP K-FOLD TRAINING (Binary)
# =========================================================
HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

gkf = GroupKFold(n_splits=5)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in gkf.split(X_seq, y, groups=patient_ids):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    
    # Compute class weights to handle imbalance
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    # Confusion matrix
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 7. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA
# =========================================================
csv_path = r"D:/fused_with_patient_id.csv"
chunksize = 10000
label_column = "class"
scan_column = "Image Data ID"  # use this to group slices of same scan

X_list = []
y_list = []
scan_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    # Drop non-numeric columns except label and scan ID
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col not in [label_column, scan_column]]
    chunk = chunk.drop(columns=cols_to_drop)
    
    # Features
    X_chunk = chunk.drop(columns=[label_column, scan_column]).values.astype('float32')
    
    # Labels and scan IDs
    y_chunk = chunk[label_column].values
    s_chunk = chunk[scan_column].values
    
    X_list.append(X_chunk)
    y_list.append(y_chunk)
    scan_list.append(s_chunk)

# Concatenate all chunks
X = np.vstack(X_list)
y_raw = np.concatenate(y_list)
scan_ids = np.concatenate(scan_list)

# =========================================================
# 2. FILTER TO AD AND MCI ONLY
# =========================================================
mask = np.isin(y_raw, ["AD", "MCI"])
X = X[mask]
y_raw = y_raw[mask]
scan_ids = scan_ids[mask]

# Map labels: AD=0, MCI=1
y = np.array([0 if label=="AD" else 1 for label in y_raw])
class_names = ["AD", "MCI"]

print("Filtered features shape:", X.shape)
print("Unique scans:", len(np.unique(scan_ids)))
print("Number of classes:", len(class_names))

# =========================================================
# 3. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN*FEAT_PER_STEP]  # trim excess
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 4. AGGREGATE SLICES PER SCAN (OPTIONAL SAFETY)
# =========================================================
# Aggregate slices per scan to avoid slice leakage
df = pd.DataFrame(X_seq.reshape(X_seq.shape[0], -1))
df['scan_id'] = scan_ids
df['label'] = y

# Group by scan
df_agg = df.groupby('scan_id').mean()
y_agg = df.groupby('scan_id')['label'].first().values
scan_ids_agg = df.groupby('scan_id').first().index.values

X_seq = df_agg.values.reshape(df_agg.shape[0], SEQ_LEN, FEAT_PER_STEP)
y = y_agg
scan_ids = scan_ids_agg
print("After aggregation: X_seq shape:", X_seq.shape)

# =========================================================
# 5. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 6. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, weights

# =========================================================
# 7. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 8. TRAIN/TEST SPLIT BY SCAN (NO SLICE LEAKAGE)
# =========================================================
from sklearn.model_selection import GroupKFold

HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

gkf = GroupKFold(n_splits=5)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in gkf.split(X_seq, y, groups=scan_ids):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    
    # Class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 9. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# =========================================================
# 0. IMPORTS
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# =========================================================
# 1. LOAD CSV IN CHUNKS AND PREPARE DATA
# =========================================================
csv_path = r"D:/fused_with_patient_id.csv"
chunksize = 10000
label_column = "class"
scan_column = "Image Data ID"  # each slice belongs to a scan

X_list = []
y_list = []
scan_list = []

for chunk in pd.read_csv(csv_path, chunksize=chunksize):
    # Drop non-numeric columns except label and scan ID
    non_numeric_cols = chunk.select_dtypes(include=['object']).columns.tolist()
    cols_to_drop = [col for col in non_numeric_cols if col not in [label_column, scan_column]]
    chunk = chunk.drop(columns=cols_to_drop)
    
    # Features
    X_chunk = chunk.drop(columns=[label_column, scan_column]).values.astype('float32')
    
    # Labels and scan IDs
    y_chunk = chunk[label_column].values
    s_chunk = chunk[scan_column].values
    
    X_list.append(X_chunk)
    y_list.append(y_chunk)
    scan_list.append(s_chunk)

# Concatenate all chunks
X = np.vstack(X_list)
y_raw = np.concatenate(y_list)
scan_ids = np.concatenate(scan_list)

# =========================================================
# 2. FILTER TO AD AND MCI ONLY
# =========================================================
mask = np.isin(y_raw, ["AD", "MCI"])
X = X[mask]
y_raw = y_raw[mask]
scan_ids = scan_ids[mask]

# Map labels: AD=0, MCI=1
y = np.array([0 if label=="AD" else 1 for label in y_raw])
class_names = ["AD", "MCI"]

print("Filtered features shape:", X.shape)
print("Unique scans:", len(np.unique(scan_ids)))
print("Number of classes:", len(class_names))

# =========================================================
# 3. RESHAPE FOR LSTM
# =========================================================
SEQ_LEN = 25
FEAT_PER_STEP = X.shape[1] // SEQ_LEN
X = X[:, :SEQ_LEN * FEAT_PER_STEP]  # trim excess
X_seq = X.reshape(X.shape[0], SEQ_LEN, FEAT_PER_STEP)
print("LSTM input shape:", X_seq.shape)

# =========================================================
# 4. DATASET CLASS
# =========================================================
class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

# =========================================================
# 5. MODEL DEFINITION
# =========================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    
    def forward(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

class LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = AttentionLayer(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, weights = self.attn(lstm_out)
        context = self.dropout(context)
        out = self.fc(context)
        return out, weights

# =========================================================
# 6. TRAIN & EVAL FUNCTIONS
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

def evaluate_model(model, dataloader):
    model.eval()
    preds_list = []
    true_list = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs, _ = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_list.append(preds)
            true_list.append(labels.cpu().numpy())
    return np.concatenate(preds_list), np.concatenate(true_list)

# =========================================================
# 7. GROUP K-FOLD BY SCAN (NO SLICE LEAKAGE)
# =========================================================
from sklearn.model_selection import GroupKFold

HIDDEN = 64
NUM_CLASSES = len(class_names)
EPOCHS = 10
BATCH_SIZE = 32

gkf = GroupKFold(n_splits=5)
fold = 1
all_conf_matrices = []
all_preds = []
all_true = []

for train_idx, test_idx in gkf.split(X_seq, y, groups=scan_ids):
    print(f"\n================ Fold {fold} ================")
    
    X_train, X_test = X_seq[train_idx], X_seq[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = LSTMDataset(X_train, y_train)
    test_dataset = LSTMDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LSTM_Attention(FEAT_PER_STEP, HIDDEN, NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0008)
    
    # Class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Train
    train_model(model, criterion, optimizer, train_loader, epochs=EPOCHS)
    
    # Evaluate
    preds, true = evaluate_model(model, test_loader)
    
    all_preds.append(preds)
    all_true.append(true)
    
    cm = confusion_matrix(true, preds)
    all_conf_matrices.append(cm)
    
    acc = accuracy_score(true, preds)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(f"Classification Report:\n{classification_report(true, preds, target_names=class_names)}")
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.show()
    
    fold += 1

# =========================================================
# 8. AGGREGATED METRICS
# =========================================================
all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

print("Overall Accuracy:", accuracy_score(all_true, all_preds))
print("Overall Classification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

sum_cm = np.sum(all_conf_matrices, axis=0)
plt.figure(figsize=(6,5))
sns.heatmap(sum_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Aggregated Confusion Matrix (5 folds)")
plt.show()


In [ ]:
# apply bert model for textual embeddings


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm


In [ ]:
csv_path = r"D:\PD_1YEAR\ADNI1_Merged_MRI_Metadata.csv"
df = pd.read_csv(csv_path)

print(df.columns)


In [ ]:
def row_to_text(row):
    text = (
        f"Subject sex is {row['Sex']}. "
        f"Age is {int(row['Age'])} years. "
        f"Clinical group is {row['Group']}. "
        f"Visit code is {row['VISCODE']}. "
        f"Imaging modality is {row['Modality']}. "
        f"Scan description is {row['Description']}. "
        f"Scan type is {row['Type']}."
    )
    return text


In [ ]:
df["text"] = df.apply(row_to_text, axis=1)
df["text"].iloc[0]


In [ ]:
symptom_map = {
    "AXNAUSEA": "nausea",
    "AXVOMIT": "vomiting",
    "AXDIARRH": "diarrhea",
    "AXCONSTP": "constipation",
    "AXABDOMN": "abdominal pain",
    "AXSWEATN": "excessive sweating",
    "AXDIZZY": "dizziness",
    "AXENERGY": "low energy",
    "AXDROWSY": "drowsiness",
    "AXVISION": "vision problems",
    "AXHDACHE": "headache",
    "AXDRYMTH": "dry mouth",
    "AXBREATH": "shortness of breath",
    "AXCOUGH": "cough",
    "AXPALPIT": "palpitations",
    "AXCHEST": "chest discomfort",
    "AXURNDIS": "urinary discomfort",
    "AXURNFRQ": "frequent urination",
    "AXANKLE": "ankle swelling",
    "AXMUSCLE": "muscle pain",
    "AXRASH": "skin rash",
    "AXINSOMN": "insomnia",
    "AXDPMOOD": "depressed mood",
    "AXCRYING": "crying spells",
    "AXELMOOD": "elevated mood",
    "AXWANDER": "wandering behavior",
    "AXFALL": "history of falls",
    "AXOTHER": "other symptoms"
}


In [ ]:
def symptoms_to_text(row):
    present_symptoms = []

    for col, desc in symptom_map.items():
        if col in row and row[col] == 1:
            present_symptoms.append(desc)

    if len(present_symptoms) == 0:
        return "No significant symptoms reported."
    else:
        return "Reported symptoms include " + ", ".join(present_symptoms) + "."


In [ ]:
def row_to_text(row):
    text = (
        f"Subject sex is {'Male' if row['Sex']=='M' else 'Female'}. "
        f"Age is {int(row['Age'])} years. "
        f"Visit is {row['VISCODE']}. "
        f"Imaging modality is MRI. "
        f"Scan description is {row['Description']}. "
        f"Scan type is {row['Type']}. "
    )

    text += symptoms_to_text(row)

    return text


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import numpy as np

# ===============================
# 1️⃣ Load CSV
# ===============================
csv_path = r"D:\PD_1YEAR\ADNI1_Merged_MRI_Metadata.csv"
df = pd.read_csv(csv_path)

# ===============================
# 2️⃣ Map symptom columns
# ===============================
symptom_map = {
    "AXNAUSEA": "nausea",
    "AXVOMIT": "vomiting",
    "AXDIARRH": "diarrhea",
    "AXCONSTP": "constipation",
    "AXABDOMN": "abdominal pain",
    "AXSWEATN": "excessive sweating",
    "AXDIZZY": "dizziness",
    "AXENERGY": "low energy",
    "AXDROWSY": "drowsiness",
    "AXVISION": "vision problems",
    "AXHDACHE": "headache",
    "AXDRYMTH": "dry mouth",
    "AXBREATH": "shortness of breath",
    "AXCOUGH": "cough",
    "AXPALPIT": "palpitations",
    "AXCHEST": "chest discomfort",
    "AXURNDIS": "urinary discomfort",
    "AXURNFRQ": "frequent urination",
    "AXANKLE": "ankle swelling",
    "AXMUSCLE": "muscle pain",
    "AXRASH": "skin rash",
    "AXINSOMN": "insomnia",
    "AXDPMOOD": "depressed mood",
    "AXCRYING": "crying spells",
    "AXELMOOD": "elevated mood",
    "AXWANDER": "wandering behavior",
    "AXFALL": "history of falls",
    "AXOTHER": "other symptoms"
}

# ===============================
# 3️⃣ Convert symptoms → text
# ===============================
def symptoms_to_text(row):
    present_symptoms = [desc for col, desc in symptom_map.items() if col in row and row[col] == 1]
    if len(present_symptoms) == 0:
        return "No significant symptoms reported."
    else:
        return "Reported symptoms include " + ", ".join(present_symptoms) + "."

# ===============================
# 4️⃣ Build full text per row
# ===============================
def row_to_text(row):
    text = (
        f"Subject sex is {'Male' if row['Sex']=='M' else 'Female'}. "
        f"Age is {int(row['Age'])} years. "
        f"Visit is {row['VISCODE']}. "
        f"Imaging modality is MRI. "
        f"Scan description is {row['Description']}. "
        f"Scan type is {row['Type']}. "
    )
    text += symptoms_to_text(row)
    return text

df["text"] = df.apply(row_to_text, axis=1)

# ===============================
# 5️⃣ Load BERT
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model = model.to(device)
model.eval()

# ===============================
# 6️⃣ Generate embeddings
# ===============================
def get_bert_embeddings(texts, batch_size=16):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)
            outputs = model(**encoded)
            cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)

text_embeddings = get_bert_embeddings(df["text"].tolist())
print("BERT embeddings shape:", text_embeddings.shape)  # (num_subjects, 768)

# ===============================
# 7️⃣ Optional: reduce to 512 for fusion
# ===============================
import torch.nn as nn
bert_fc = nn.Linear(768, 512)
bert_512 = bert_fc(text_embeddings)

# ===============================
# 8️⃣ Save embeddings
# ===============================
np.save("bert_text_embeddings_512.npy", bert_512.numpy())


In [ ]:
import pandas as pd

# ===============================
# 1️⃣ Load CSV
# ===============================
csv_path = r"D:\PD_1YEAR\ADNI1_Merged_MRI_Metadata.csv"
df = pd.read_csv(csv_path)

# ===============================
# 2️⃣ Symptom mapping
# ===============================
symptom_map = {
    "AXNAUSEA": "nausea",
    "AXVOMIT": "vomiting",
    "AXDIARRH": "diarrhea",
    "AXCONSTP": "constipation",
    "AXABDOMN": "abdominal pain",
    "AXSWEATN": "excessive sweating",
    "AXDIZZY": "dizziness",
    "AXENERGY": "low energy",
    "AXDROWSY": "drowsiness",
    "AXVISION": "vision problems",
    "AXHDACHE": "headache",
    "AXDRYMTH": "dry mouth",
    "AXBREATH": "shortness of breath",
    "AXCOUGH": "cough",
    "AXPALPIT": "palpitations",
    "AXCHEST": "chest discomfort",
    "AXURNDIS": "urinary discomfort",
    "AXURNFRQ": "frequent urination",
    "AXANKLE": "ankle swelling",
    "AXMUSCLE": "muscle pain",
    "AXRASH": "skin rash",
    "AXINSOMN": "insomnia",
    "AXDPMOOD": "depressed mood",
    "AXCRYING": "crying spells",
    "AXELMOOD": "elevated mood",
    "AXWANDER": "wandering behavior",
    "AXFALL": "history of falls",
    "AXOTHER": "other symptoms"
}

# ===============================
# 3️⃣ Convert symptom flags → text
# ===============================
def symptoms_to_text(row):
    present_symptoms = [desc for col, desc in symptom_map.items() if col in row and row[col] == 1]
    if len(present_symptoms) == 0:
        return "No significant symptoms reported."
    else:
        return "Reported symptoms include " + ", ".join(present_symptoms) + "."

# ===============================
# 4️⃣ Create clean text prompt (demographics + symptoms only)
# ===============================
def row_to_text(row):
    text = (
        f"Subject sex is {'Male' if row['Sex']=='M' else 'Female'}. "
        f"Age is {int(row['Age'])} years. "
    )
    text += symptoms_to_text(row)
    return text

df["text_prompt"] = df.apply(row_to_text, axis=1)

# ===============================
# 5️⃣ Keep only relevant columns
# ===============================
columns_to_keep = ["Image Data ID", "RID", "PTID", "Group", "text_prompt"]
df_clean = df[columns_to_keep]

# ===============================
# 6️⃣ Save cleaned CSV
# ===============================
output_path = "ADNI_text_prompts_cleaned.csv"
df_clean.to_csv(output_path, index=False)
print("Cleaned CSV saved at:", output_path)
print("CSV shape:", df_clean.shape)


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import numpy as np
import torch.nn as nn

# ===============================
# 1️⃣ Load cleaned CSV
# ===============================
csv_path = "ADNI_text_prompts_cleaned.csv"
df = pd.read_csv(csv_path)

# Make sure text_prompt column exists
assert "text_prompt" in df.columns, "text_prompt column not found!"

# ===============================
# 2️⃣ Setup BERT
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model = model.to(device)
model.eval()  # Important: evaluation mode

# ===============================
# 3️⃣ Function to generate embeddings
# ===============================
def get_bert_embeddings(texts, batch_size=16):
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]

            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            outputs = model(**encoded)

            # CLS token embedding
            cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: (B, 768)
            all_embeddings.append(cls_emb.cpu())

    return torch.cat(all_embeddings, dim=0)

# ===============================
# 4️⃣ Generate BERT embeddings
# ===============================
text_embeddings = get_bert_embeddings(df["text_prompt"].tolist())
print("BERT embeddings shape:", text_embeddings.shape)  # (num_subjects, 768)

# ===============================
# 5️⃣ Reduce 768 → 512 dimensions (for fusion)
# ===============================
bert_fc = nn.Linear(768, 512)
bert_512 = bert_fc(text_embeddings)

# Optional: normalize embeddings
bert_512 = torch.nn.functional.normalize(bert_512, p=2, dim=1)

# ===============================
# 6️⃣ Save embeddings
# ===============================
# Detach tensor before converting to NumPy
bert_numpy = bert_512.detach().cpu().numpy()

# Save as NumPy array
np.save("ADNI_BERT_embeddings_512.npy", bert_numpy)

# Optional: save with IDs in CSV
import pandas as pd
bert_df = pd.DataFrame(bert_numpy, columns=[f"bert_{i}" for i in range(512)])
final_df = pd.concat([df[["Image Data ID", "RID", "PTID", "Group"]], bert_df], axis=1)
final_df.to_csv("ADNI_BERT_embeddings_512.csv", index=False)

print("BERT embeddings saved! Shape:", bert_numpy.shape)



In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# ===============================
# 1️⃣ Load MRI embeddings
# ===============================
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"
mri_df = pd.read_csv(mri_csv_path)

# Assuming the first few columns are IDs: keep them separately
id_cols = ["Image Data ID", "RID", "PTID", "Group"]
feature_cols = [col for col in mri_df.columns if col not in id_cols]

mri_features = mri_df[feature_cols].values
print("Original MRI features shape:", mri_features.shape)

# ===============================
# 2️⃣ Standardize MRI features
# ===============================
scaler = StandardScaler()
mri_features_scaled = scaler.fit_transform(mri_features)

# Convert to PyTorch tensor
mri_tensor = torch.tensor(mri_features_scaled, dtype=torch.float32)

# ===============================
# 3️⃣ Reduce MRI dimension to 512
# ===============================
input_dim = mri_tensor.shape[1]  # original MRI embedding dim
output_dim = 512

linear_mri = nn.Linear(input_dim, output_dim)
# Pass through linear layer
mri_512 = linear_mri(mri_tensor)
mri_512 = torch.nn.functional.normalize(mri_512, p=2, dim=1)  # normalize

# ===============================
# 4️⃣ Load BERT embeddings
# ===============================
bert_csv_path = "ADNI_BERT_embeddings_512.csv"
bert_df = pd.read_csv(bert_csv_path)

# Make sure IDs match
assert all(mri_df["Image Data ID"] == bert_df["Image Data ID"]), "Mismatch in IDs!"

bert_features = bert_df[[f"bert_{i}" for i in range(512)]].values
bert_tensor = torch.tensor(bert_features, dtype=torch.float32)

# ===============================
# 5️⃣ Fuse MRI + BERT embeddings
# ===============================
fused_tensor = torch.cat([mri_512, bert_tensor], dim=1)  # shape: (N, 1024)
fused_tensor = fused_tensor.detach().cpu().numpy()  # detach from graph

# ===============================
# 6️⃣ Save fused CSV with IDs
# ===============================
fused_df = pd.DataFrame(fused_tensor, columns=[f"feat_{i}" for i in range(fused_tensor.shape[1])])
fused_df = pd.concat([mri_df[id_cols].reset_index(drop=True), fused_df], axis=1)

fused_df.to_csv("ADNI_MRI_BERT_fused_1024.csv", index=False)
print("Fused CSV saved! Shape:", fused_df.shape)


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# ===============================
# 1️⃣ Load MRI embeddings
# ===============================
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"
mri_df = pd.read_csv(mri_csv_path)

# ===============================
# 2️⃣ Separate ID columns from features
# ===============================
id_cols = ["Image Data ID", "RID", "PTID", "Group"]  # adjust if needed
feature_cols = [col for col in mri_df.columns if col not in id_cols]

# Ensure numeric
mri_features = mri_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
print("Original MRI features shape:", mri_features.shape)

# ===============================
# 3️⃣ Standardize MRI features
# ===============================
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features)

# ===============================
# 4️⃣ PCA reduction to 512 dimensions
# ===============================
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)
print("MRI PCA features shape:", mri_pca.shape)

# Convert to tensor and normalize
mri_tensor = torch.tensor(mri_pca, dtype=torch.float32)
mri_tensor = torch.nn.functional.normalize(mri_tensor, p=2, dim=1)

# ===============================
# 5️⃣ Load BERT embeddings
# ===============================
bert_csv_path = "ADNI_BERT_embeddings_512.csv"
bert_df = pd.read_csv(bert_csv_path)

# Make sure IDs match
assert all(mri_df["Image Data ID"] == bert_df["Image Data ID"]), "Mismatch in IDs!"

bert_features = bert_df[[f"bert_{i}" for i in range(512)]].values
bert_tensor = torch.tensor(bert_features, dtype=torch.float32)

# ===============================
# 6️⃣ Fuse MRI + BERT embeddings
# ===============================
fused_tensor = torch.cat([mri_tensor, bert_tensor], dim=1)  # 512 + 512 = 1024
fused_tensor = fused_tensor.detach().cpu().numpy()

# ===============================
# 7️⃣ Save fused CSV with IDs
# ===============================
fused_df = pd.DataFrame(fused_tensor, columns=[f"feat_{i}" for i in range(fused_tensor.shape[1])])
fused_df = pd.concat([mri_df[id_cols].reset_index(drop=True), fused_df], axis=1)

fused_df.to_csv("ADNI_MRI_BERT_fused_1024_PCA.csv", index=False)
print("Fused CSV saved! Shape:", fused_df.shape)


In [ ]:
print(mri_df.columns.tolist())


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# ===============================
# 1️⃣ Load MRI embeddings
# ===============================
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"
mri_df = pd.read_csv(mri_csv_path)

# ===============================
# 2️⃣ Identify ID and feature columns
# ===============================
id_cols = ["filename", "class"]  # ID + label
feature_cols = [col for col in mri_df.columns if col not in id_cols]

# Ensure features are numeric
mri_features = mri_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
print("Original MRI features shape:", mri_features.shape)

# ===============================
# 3️⃣ Standardize MRI features
# ===============================
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features)

# ===============================
# 4️⃣ PCA reduction to 512
# ===============================
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)
print("MRI PCA features shape:", mri_pca.shape)

# Normalize and convert to tensor
mri_tensor = torch.tensor(mri_pca, dtype=torch.float32)
mri_tensor = torch.nn.functional.normalize(mri_tensor, p=2, dim=1)

# ===============================
# 5️⃣ Load BERT embeddings
# ===============================
bert_csv_path = "ADNI_BERT_embeddings_512.csv"
bert_df = pd.read_csv(bert_csv_path)

# Strip column names just in case
mri_df.columns = mri_df.columns.str.strip()
bert_df.columns = bert_df.columns.str.strip()

# Use filename to match IDs
assert all(mri_df["filename"] == bert_df["filename"]), "Mismatch in filenames!"

bert_features = bert_df[[f"bert_{i}" for i in range(512)]].values
bert_tensor = torch.tensor(bert_features, dtype=torch.float32)

# ===============================
# 6️⃣ Fuse MRI + BERT embeddings
# ===============================
fused_tensor = torch.cat([mri_tensor, bert_tensor], dim=1)  # 512 + 512 = 1024
fused_tensor = fused_tensor.detach().cpu().numpy()

# ===============================
# 7️⃣ Save fused CSV with filename and class
# ===============================
fused_df = pd.DataFrame(fused_tensor, columns=[f"feat_{i}" for i in range(fused_tensor.shape[1])])
fused_df = pd.concat([mri_df[id_cols].reset_index(drop=True), fused_df], axis=1)

fused_df.to_csv("ADNI_MRI_BERT_fused_1024_PCA.csv", index=False)
print("Fused CSV saved! Shape:", fused_df.shape)


In [ ]:
import pandas as pd

# Sample filenames
features_df = pd.DataFrame({
    "filename": [
        "002_S_0619_I118678_AD_slice_0079.png",
        "003_S_0620_I118679_CN_slice_0010.png"
    ]
})

# Extract ID pattern like I118678
features_df["Image Data ID"] = features_df["filename"].str.extract(r"(I\d+)")
print(features_df)


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -------------------------------
# 1️⃣ Load MRI embeddings
# -------------------------------
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"
mri_df = pd.read_csv(mri_csv_path)

# -------------------------------
# 2️⃣ Extract Image Data ID from filename (if not already present)
# -------------------------------
if "Image Data ID" not in mri_df.columns:
    mri_df["Image Data ID"] = mri_df["filename"].str.extract(r"(I\d+)")

# Keep label column
id_cols = ["Image Data ID", "class"]

# Select numeric MRI feature columns only
feature_cols = [col for col in mri_df.columns if col not in id_cols + ["filename"]]
mri_features = mri_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
print("Original MRI features shape:", mri_features.shape)

# -------------------------------
# 3️⃣ Standardize MRI features
# -------------------------------
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features)

# -------------------------------
# 4️⃣ PCA reduction to 512 dims
# -------------------------------
pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Normalize
mri_tensor = torch.tensor(mri_pca, dtype=torch.float32)
mri_tensor = torch.nn.functional.normalize(mri_tensor, p=2, dim=1)

# -------------------------------
# 5️⃣ Load BERT embeddings
# -------------------------------
bert_csv_path = "ADNI_BERT_embeddings_512.csv"
bert_df = pd.read_csv(bert_csv_path)

# -------------------------------
# 6️⃣ Merge MRI PCA features with BERT embeddings using Image Data ID
# -------------------------------
merged_df = mri_df[id_cols].merge(
    pd.DataFrame(mri_tensor.numpy(), columns=[f"mri_{i}" for i in range(512)]).join(mri_df[id_cols]),
    on="Image Data ID"
)

# Align BERT embeddings
bert_feature_cols = [f"bert_{i}" for i in range(512)]
bert_features_df = bert_df[["Image Data ID"] + bert_feature_cols]

# Merge MRI PCA features with BERT embeddings
fused_df = merged_df.merge(bert_features_df, on="Image Data ID")
print("Fused DataFrame shape:", fused_df.shape)

# -------------------------------
# 7️⃣ Save fused CSV
# -------------------------------
fused_df.to_csv("ADNI_MRI_BERT_fused_1024.csv", index=False)
print("Fused CSV saved successfully!")


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -------------------------------
# 1️⃣ Load MRI embeddings
# -------------------------------
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"
mri_df = pd.read_csv(mri_csv_path)

# Extract Image Data ID from filename if needed
if "Image Data ID" not in mri_df.columns:
    mri_df["Image Data ID"] = mri_df["filename"].str.extract(r"(I\d+)")

# Keep IDs and class
ids = mri_df[["Image Data ID", "class"]].copy()

# Select numeric MRI feature columns
feature_cols = [col for col in mri_df.columns if col not in ["filename", "Image Data ID", "class"]]
mri_features = mri_df[feature_cols].astype(float).values
print("Original MRI features shape:", mri_features.shape)

# -------------------------------
# 2️⃣ Standardize + PCA
# -------------------------------
scaler = StandardScaler()
mri_scaled = scaler.fit_transform(mri_features)

pca = PCA(n_components=512, random_state=42)
mri_pca = pca.fit_transform(mri_scaled)

# Normalize
mri_pca = torch.tensor(mri_pca, dtype=torch.float32)
mri_pca = torch.nn.functional.normalize(mri_pca, p=2, dim=1)
mri_pca = mri_pca.numpy()  # keep as NumPy for memory efficiency

# -------------------------------
# 3️⃣ Load BERT embeddings
# -------------------------------
bert_csv_path = "ADNI_BERT_embeddings_512.csv"
bert_df = pd.read_csv(bert_csv_path)

# Make sure IDs match exactly
bert_df = bert_df.set_index("Image Data ID").loc[ids["Image Data ID"]]
bert_features = bert_df[[f"bert_{i}" for i in range(512)]].values

# -------------------------------
# 4️⃣ Fuse MRI PCA + BERT embeddings
# -------------------------------
# Shape: (num_samples, 1024)
fused_features = np.hstack([mri_pca, bert_features])
print("Fused features shape:", fused_features.shape)

# -------------------------------
# 5️⃣ Save fused CSV
# -------------------------------
fused_df = pd.DataFrame(
    fused_features,
    columns=[f"mri_{i}" for i in range(512)] + [f"bert_{i}" for i in range(512)]
)
# Add back IDs and class
fused_df.insert(0, "class", ids["class"].values)
fused_df.insert(0, "Image Data ID", ids["Image Data ID"].values)

fused_df.to_csv("ADNI_MRI_BERT_fused_1024.csv", index=False)
print("Fused CSV saved successfully!")


In [ ]:
import pandas as pd
import numpy as np
import torch

# -------------------------------
# 1️⃣ Load CSVs
# -------------------------------
mri_df = pd.read_csv("resedualnetwork50_selected_features_cumulative.csv")      # replace with your path
bert_df = pd.read_csv("ADNI_BERT_embeddings_512.csv")    # replace with your path

# -------------------------------
# 2️⃣ Keep only common IDs
# -------------------------------
common_ids = set(mri_df["Image Data ID"]).intersection(set(bert_df["Image Data ID"]))
print(f"Number of common IDs: {len(common_ids)}")

# Filter and sort both dataframes
mri_df = mri_df[mri_df["Image Data ID"].isin(common_ids)].sort_values("Image Data ID").reset_index(drop=True)
bert_df = bert_df[bert_df["Image Data ID"].isin(common_ids)].sort_values("Image Data ID").reset_index(drop=True)

# -------------------------------
# 3️⃣ Convert features to numpy arrays
# -------------------------------
# Select only feature columns (assuming MRI features are named like mri_0, mri_1, ...)
mri_feature_cols = [col for col in mri_df.columns if col.startswith("mri_")]
bert_feature_cols = [col for col in bert_df.columns if col.startswith("bert_")]

mri_features = mri_df[mri_feature_cols].values.astype(np.float32)
bert_features = bert_df[bert_feature_cols].values.astype(np.float32)

print(f"MRI features shape: {mri_features.shape}")
print(f"BERT features shape: {bert_features.shape}")

# -------------------------------
# 4️⃣ Fuse features efficiently
# -------------------------------
# Concatenate along feature axis
fused_features = np.concatenate([mri_features, bert_features], axis=1)
print(f"Fused feature shape: {fused_features.shape}")

# Convert to torch tensor (if using PyTorch)
fused_tensor = torch.tensor(fused_features, dtype=torch.float32)
print(f"Fused tensor shape: {fused_tensor.shape}")

# -------------------------------
# 5️⃣ Optional: Save fused features to CSV
# -------------------------------
fused_df = pd.DataFrame(fused_features, columns=mri_feature_cols + bert_feature_cols)
fused_df["Image Data ID"] = mri_df["Image Data ID"]
fused_df.to_csv("fused_mri_bert_features.csv", index=False)


In [ ]:
# See all columns exactly as pandas sees them
print(mri_df.columns.tolist())
print(bert_df.columns.tolist())


In [ ]:
# Extract subject ID from filename (adjust regex to match your filenames)
mri_df['subject_id'] = mri_df['filename'].str.extract(r'(I\d+)_')[0]

# Now you can merge or take intersection with BERT CSV
common_ids = set(mri_df['subject_id']).intersection(set(bert_df['Image Data ID']))
print(f"Number of common IDs: {len(common_ids)}")

# Filter MRI and BERT dataframes to keep only common IDs
mri_df_filtered = mri_df[mri_df['subject_id'].isin(common_ids)]
bert_df_filtered = bert_df[bert_df['Image Data ID'].isin(common_ids)]


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

# -------------------------------
# 1️⃣ Load CSV files
# -------------------------------
mri_csv_path = "resedualnetwork50_selected_features_cumulative.csv"          # Your MRI CSV path
bert_csv_path = "ADNI_BERT_embeddings_512.csv"  # Your BERT CSV path

mri_df = pd.read_csv(mri_csv_path)
bert_df = pd.read_csv(bert_csv_path)

# -------------------------------
# 2️⃣ Reduce MRI features to 512D using PCA
# -------------------------------
# Select only MRI feature columns (exclude 'filename' and 'class')
mri_feature_cols = [col for col in mri_df.columns if col not in ['filename', 'class']]
mri_features = mri_df[mri_feature_cols].values

# Apply PCA
pca = PCA(n_components=512)
mri_features_512d = pca.fit_transform(mri_features)

# Save MRI 512D features CSV
mri_512_df = pd.DataFrame(mri_features_512d)
mri_512_df['filename'] = mri_df['filename']
mri_512_df['label'] = mri_df['class']
mri_512_df.to_csv("mri_features_512d.csv", index=False)
print("MRI 512D CSV saved successfully!")

# -------------------------------
# 3️⃣ Extract Image Data ID from filename
# -------------------------------
# Assuming filename format: "I31143_AD_axial_55.png"
mri_512_df['Image Data ID'] = mri_512_df['filename'].apply(lambda x: x.split('_')[0])
mri_512_df = mri_512_df.drop(columns=['filename'])

# -------------------------------
# 4️⃣ Keep only common IDs
# -------------------------------
common_ids = set(mri_512_df['Image Data ID']).intersection(set(bert_df['Image Data ID']))
print(f"Number of common IDs: {len(common_ids)}")

mri_filtered = mri_512_df[mri_512_df['Image Data ID'].isin(common_ids)].sort_values('Image Data ID')
bert_filtered = bert_df[bert_df['Image Data ID'].isin(common_ids)].sort_values('Image Data ID')

# -------------------------------
# 5️⃣ Concatenate MRI and BERT features
# -------------------------------
# Drop columns not needed for features
bert_features = bert_filtered.drop(columns=['Image Data ID', 'RID', 'PTID', 'Group']).values
mri_features_final = mri_filtered.drop(columns=['Image Data ID', 'label']).values

fused_features = np.concatenate([mri_features_final, bert_features], axis=1)
labels = mri_filtered['label'].values

# -------------------------------
# 6️⃣ Save fused CSV
# -------------------------------
fused_df = pd.DataFrame(fused_features)
fused_df['label'] = labels

# Optional: add column names
num_mri_feats = mri_features_final.shape[1]
num_text_feats = bert_features.shape[1]

mri_cols = [f'mri_{i}' for i in range(num_mri_feats)]
text_cols = [f'text_{i}' for i in range(num_text_feats)]
fused_df.columns = mri_cols + text_cols + ['label']

fused_df.to_csv("fused_mri_text_features.csv", index=False)
print("Fused CSV saved successfully!")


In [ ]:
print("MRI IDs sample:", mri_512_df['Image Data ID'].head(10).tolist())
print("BERT IDs sample:", bert_df['Image Data ID'].head(10).tolist())


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

# -----------------------------
# 1️⃣ Load CSVs
# -----------------------------
mri_df = pd.read_csv("resedualnetwork50_selected_features_cumulative.csv")      # replace with your path
bert_df = pd.read_csv("ADNI_BERT_embeddings_512.csv")  # replace with your path

# -----------------------------
# 2️⃣ Reduce MRI features to 512D using PCA
# -----------------------------
# Select only MRI features (exclude 'filename', 'class')
feature_cols = [col for col in mri_df.columns if col not in ['filename', 'class']]
pca = PCA(n_components=512)
mri_features_512 = pca.fit_transform(mri_df[feature_cols])

# Create a new dataframe
mri_512_df = pd.DataFrame(mri_features_512, columns=[f'mri_{i}' for i in range(512)])
mri_512_df['filename'] = mri_df['filename']
mri_512_df['class'] = mri_df['class']

# Save MRI 512D CSV
mri_512_df.to_csv("MRI_512D_features.csv", index=False)
print("MRI 512D CSV saved successfully!")

# -----------------------------
# 3️⃣ Extract subject ID from filename
# -----------------------------
# Adjust regex based on your filenames: I31143_AD_axial_55.png -> I31143
mri_512_df['subject_id'] = mri_512_df['filename'].str.extract(r'(I\d+)_')[0]

# -----------------------------
# 4️⃣ Find common IDs with BERT embeddings
# -----------------------------
common_ids = set(mri_512_df['subject_id']).intersection(set(bert_df['Image Data ID']))
print(f"Number of common IDs: {len(common_ids)}")

# Filter MRI and BERT dataframes to keep only common IDs
mri_filtered = mri_512_df[mri_512_df['subject_id'].isin(common_ids)].copy()
bert_filtered = bert_df[bert_df['Image Data ID'].isin(common_ids)].copy()

# -----------------------------
# 5️⃣ Merge MRI and BERT embeddings
# -----------------------------
# Ensure BERT columns exclude 'Image Data ID', 'RID', 'PTID', 'Group'
bert_features_cols = [col for col in bert_filtered.columns if col not in ['Image Data ID', 'RID', 'PTID', 'Group']]
fused_df = pd.concat([mri_filtered.reset_index(drop=True), bert_filtered[bert_features_cols].reset_index(drop=True)], axis=1)

# -----------------------------
# 6️⃣ Save fused CSV
# -----------------------------
fused_df.to_csv("Fused_MRI_BERT.csv", index=False)
print("Fused CSV saved successfully!")


In [ ]:
# Load the fused CSV
fused_df = pd.read_csv("fused_mri_bert.csv")  # replace with your fused CSV path

# Print the shape
print("Shape of fused CSV:", fused_df.shape)

# Optionally, rename/save the file with a new name
fused_df.to_csv("fused_mri_bert_final.csv", index=False)
print("Fused CSV renamed and saved as 'fused_mri_bert_final.csv'")


In [ ]:
import pandas as pd

# Load fused CSV
fused_df = pd.read_csv("fused_MRI_BERT.csv")  # replace with your fused CSV path

# Keep only necessary columns
# Assuming MRI columns are like 'f0'...'f511' and BERT columns 'bert_0'...'bert_511'
columns_to_keep = ['Image Data ID', 'class'] + \
                  [col for col in fused_df.columns if col.startswith('f')] + \
                  [col for col in fused_df.columns if col.startswith('bert')]

fused_df_cleaned = fused_df[columns_to_keep]

# Check shape
print("Shape of cleaned fused CSV:", fused_df_cleaned.shape)

# Save cleaned fused CSV
fused_df_cleaned.to_csv("fused_MRI_BERT_cleaned.csv", index=False)
print("Cleaned fused CSV saved successfully!")


In [ ]:
import pandas as pd

# Load fused CSV
fused_df = pd.read_csv("fused_MRI_BERT.csv")

# Print all column names
print(fused_df.columns.tolist())


In [ ]:
import pandas as pd

# Load fused CSV
fused_df = pd.read_csv("fused_MRI_BERT.csv")

# Rename subject_id to Image Data ID
fused_df.rename(columns={'subject_id': 'Image Data ID'}, inplace=True)

# Keep only necessary columns: ID, class, MRI features, BERT features
columns_to_keep = ['Image Data ID', 'class'] + \
                  [col for col in fused_df.columns if col.startswith('mri_')] + \
                  [col for col in fused_df.columns if col.startswith('bert_')]

fused_df_cleaned = fused_df[columns_to_keep]

# Check shape
print("Shape of cleaned fused CSV:", fused_df_cleaned.shape)

# Save cleaned fused CSV
fused_df_cleaned.to_csv("fused_MRI_BERT_cleaned.csv", index=False)
print("Cleaned fused CSV saved successfully!")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --------------------------------
# 1️⃣ Load fused CSV
# --------------------------------
df = pd.read_csv("fused_MRI_BERT_cleaned.csv")
print("Full dataset shape:", df.shape)

# --------------------------------
# 2️⃣ Columns
# --------------------------------
id_col = "Image Data ID"   # subject-level ID
label_col = "class"

# --------------------------------
# 3️⃣ Get unique subjects
# --------------------------------
unique_subjects = df[id_col].unique()
print("Total unique subjects:", len(unique_subjects))

# --------------------------------
# 4️⃣ Subject-level split
# --------------------------------
train_subjects, test_subjects = train_test_split(
    unique_subjects,
    test_size=0.2,
    random_state=42,
    stratify=df.drop_duplicates(id_col)[label_col]
)

# --------------------------------
# 5️⃣ Slice-level filtering
# --------------------------------
train_df = df[df[id_col].isin(train_subjects)].reset_index(drop=True)
test_df  = df[df[id_col].isin(test_subjects)].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

# --------------------------------
# 6️⃣ Separate features and labels
# --------------------------------
X_train = train_df.drop(columns=[id_col, label_col])
y_train = train_df[label_col]

X_test = test_df.drop(columns=[id_col, label_col])
y_test = test_df[label_col]

# --------------------------------
# 7️⃣ Scaling (NO LEAKAGE)
# --------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --------------------------------
# 8️⃣ Rebuild scaled DataFrames
# --------------------------------
train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
train_scaled_df.insert(0, id_col, train_df[id_col])
train_scaled_df.insert(1, label_col, y_train)

test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)
test_scaled_df.insert(0, id_col, test_df[id_col])
test_scaled_df.insert(1, label_col, y_test)

# --------------------------------
# 9️⃣ Save CSVs
# --------------------------------
train_scaled_df.to_csv("train_fused_MRI_BERT_scaled.csv", index=False)
test_scaled_df.to_csv("test_fused_MRI_BERT_scaled.csv", index=False)

print("Subject-level scaled train & test CSVs saved successfully!")


In [ ]:
import pandas as pd
import numpy as np

# Load fused slice-level data
df = pd.read_csv("fused_MRI_BERT_cleaned.csv")

id_col = "Image Data ID"
label_col = "class"

feature_cols = [c for c in df.columns if c.startswith("mri_") or c.startswith("bert_")]

# Mean pooling per subject
subject_df = (
    df.groupby(id_col)[feature_cols]
    .mean()
    .reset_index()
)

# Add label (same for all slices of a subject)
labels = df[[id_col, label_col]].drop_duplicates()
subject_df = subject_df.merge(labels, on=id_col)

print("Subject-level shape:", subject_df.shape)

subject_df.to_csv("subject_level_fused.csv", index=False)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("subject_level_fused.csv")

X = df.drop(columns=["Image Data ID", "class"])
y = df["class"]
ids = df["Image Data ID"]

# Subject-level split
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling (NO leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save
train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
train_df.insert(0, "Image Data ID", id_train.values)
train_df["class"] = y_train.values

test_df = pd.DataFrame(X_test_scaled, columns=X.columns)
test_df.insert(0, "Image Data ID", id_test.values)
test_df["class"] = y_test.values

train_df.to_csv("train_subject_fused_scaled.csv", index=False)
test_df.to_csv("test_subject_fused_scaled.csv", index=False)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ====================================
# 1. Load SUBJECT-LEVEL fused dataset
# ====================================
df = pd.read_csv("subject_level_fused.csv")

print("Dataset shape:", df.shape)
print("Class distribution:\n", df["class"].value_counts())

# ====================================
# 2. Encode class labels
# ====================================
le = LabelEncoder()
df["class_encoded"] = le.fit_transform(df["class"])

print("\nClass mapping:")
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls} -> {enc}")

# ====================================
# 3. Split features and labels
# ====================================
X = df.drop(columns=["Image Data ID", "class", "class_encoded"]).values
y = df["class_encoded"].values

# ====================================
# 4. Subject-level 5-Fold CV
# ====================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []

print("\n===== SUBJECT-LEVEL 5-FOLD CV =====")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # -------------------------------
    # Scaling (fit ONLY on train)
    # -------------------------------
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # -------------------------------
    # XGBoost classifier
    # -------------------------------
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # -------------------------------
    # Evaluation
    # -------------------------------
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    fold_accuracies.append(acc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(
        y_val,
        preds,
        target_names=le.classes_,
        digits=4
    ))

# ====================================
# 5. Final CV Results
# ====================================
print("\n===== FINAL RESULTS =====")
print("Fold Accuracies:", np.round(fold_accuracies, 4))
print("Mean Accuracy :", np.mean(fold_accuracies))
print("Std  Accuracy :", np.std(fold_accuracies))


In [ ]:
print(df.columns.tolist()[:30])


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ===============================
# 1️⃣ Prepare features and labels
# ===============================
mri_cols = [c for c in df.columns if c.startswith("mri_")]
bert_cols = [c for c in df.columns if c.startswith("bert")]
label_col = "class"

# Map class names to numbers
class_mapping = {"AD": 0, "CN": 1, "MCI": 2}
df["class_encoded"] = df[label_col].map(class_mapping)

X_mri   = df[mri_cols].values
X_bert  = df[bert_cols].values
X_fused = np.concatenate([X_mri, X_bert], axis=1)
y = df["class_encoded"].values

print("MRI-only shape :", X_mri.shape)
print("BERT-only shape:", X_bert.shape)
print("Fusion shape   :", X_fused.shape)

# ===============================
# 2️⃣ Function to run subject-level 5-fold CV
# ===============================
def run_cv(X, y, name):
    print(f"\n===== {name} =====")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Scale features
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        # XGBoost classifier
        model = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric="mlogloss",
            random_state=42
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        acc = accuracy_score(y_val, preds)
        fold_accs.append(acc)

        print(f"\n--- Fold {fold} ---")
        print(f"Fold {fold} Accuracy: {acc:.4f}")
        print(classification_report(y_val, preds, target_names=class_mapping.keys()))

    print(f"\nMean Accuracy for {name} : {np.mean(fold_accs):.4f}")
    print(f"Std  Accuracy for {name} : {np.std(fold_accs):.4f}")
    return fold_accs

# ===============================
# 3️⃣ Run ablation experiments
# ===============================
acc_mri   = run_cv(X_mri, y, "MRI ONLY")
acc_bert  = run_cv(X_bert, y, "BERT ONLY")
acc_fused = run_cv(X_fused, y, "MRI + BERT (FUSION)")


In [2]:
# ==========================
# 0️⃣ Imports
# ==========================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Multiply, Add
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ==========================
# 1️⃣ Load MRI features and labels
# ==========================
df_mri = pd.read_csv("mri_features_512d.csv")  # MRI features + 'label' column

# Separate features and labels
X_mri = df_mri.drop(columns=['label']).values  # MRI features (512 dims)
y_raw = df_mri['label'].values  # labels (AD/CN/MCI)

# Encode labels as numeric
le = LabelEncoder()
y = le.fit_transform(y_raw)  # AD->0, CN->1, MCI->2 (or similar)
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ==========================
# 2️⃣ Load BERT embeddings
# ==========================
X_text = pd.read_csv("ADNI_BERT_embeddings_512.csv", header=None).values  # BERT embeddings (512 dims)
assert X_mri.shape[0] == X_text.shape[0], "Number of samples in MRI and BERT embeddings must match!"

# ==========================
# 3️⃣ Scale embeddings
# ==========================
scaler_mri = StandardScaler()
X_mri_scaled = scaler_mri.fit_transform(X_mri)

scaler_text = StandardScaler()
X_text_scaled = scaler_text.fit_transform(X_text)

# ==========================
# 4️⃣ Attention-based fusion model
# ==========================
# Inputs
input_mri = Input(shape=(512,), name="MRI_Input")
input_text = Input(shape=(512,), name="Text_Input")

# MRI branch
x1 = Dense(256, activation='relu')(input_mri)
x1 = Dropout(0.3)(x1)

# Text branch
x2 = Dense(256, activation='relu')(input_text)
x2 = Dropout(0.3)(x2)

# Attention scores
att_mri = Dense(1, activation='sigmoid', name='att_mri')(x1)
att_text = Dense(1, activation='sigmoid', name='att_text')(x2)

# Apply attention
x1_att = Multiply()([x1, att_mri])
x2_att = Multiply()([x2, att_text])

# Fuse attended features
x = Add()([x1_att, x2_att])
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)  # 3 classes: AD, CN, MCI

# Model
model = Model(inputs=[input_mri, input_text], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==========================
# 5️⃣ Train the model
# ==========================
early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    [X_mri_scaled, X_text_scaled],
    y,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

# ==========================
# 6️⃣ Evaluate
# ==========================
loss, acc = model.evaluate([X_mri_scaled, X_text_scaled], y)
print(f"Fused model accuracy: {acc*100:.2f}%")

# ==========================
# 7️⃣ Attention weight visualization
# ==========================
# Model to extract attention scores
attention_model = Model(inputs=model.inputs, outputs=[att_mri, att_text])
att_mri_values, att_text_values = attention_model.predict([X_mri_scaled, X_text_scaled])
att_mri_values = att_mri_values.flatten()
att_text_values = att_text_values.flatten()

# Scatter plot of attention weights
plt.figure(figsize=(10,5))
plt.scatter(range(len(att_mri_values)), att_mri_values, label='MRI Attention', alpha=0.7)
plt.scatter(range(len(att_text_values)), att_text_values, label='Text Attention', alpha=0.7)
plt.xlabel('Sample index')
plt.ylabel('Attention weight')
plt.title('MRI vs Text Attention Weights per Sample')
plt.legend()
plt.show()

# Optional: average attention per class
for i, class_name in enumerate(le.classes_):
    mask = (y == i)
    print(f"{class_name} - Avg MRI attention: {np.mean(att_mri_values[mask]):.3f}, Avg Text attention: {np.mean(att_text_values[mask]):.3f}")


Classes mapping: {'AD': 0, 'CN': 1, 'MCI': 2}


C:\Users\Nimra Nadeem\AppData\Local\Temp\ipykernel_19852\131773517.py:30: DtypeWarning: Columns (1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254

AssertionError: Number of samples in MRI and BERT embeddings must match!

In [3]:
print("MRI features shape:", X_mri.shape)
print("BERT embeddings shape:", X_text.shape)


MRI features shape: (157312, 513)
BERT embeddings shape: (1465, 516)


In [7]:
df_mri = pd.read_csv("mri_features_512d.csv")
print(df_mri.head())
print(df_mri.dtypes)


          0         1         2         3         4         5         6  \
0  3.933728 -6.790685  0.958309 -1.087116 -1.830195 -1.364699  1.263548   
1  4.155911 -8.424635  2.562537 -1.077359 -1.533579 -1.728293  1.085525   
2  2.542780 -6.620580  0.592079 -0.605384 -0.372335 -1.212788  0.575913   
3  1.441322 -5.354963 -0.825883 -0.711870  0.313435 -0.904823  0.239762   
4  2.082709 -6.378020  0.336107 -0.308100 -0.059328 -1.104075  0.314395   

          7         8         9  ...       504       505       506       507  \
0  0.660527 -1.431475  1.493086  ... -0.008533  0.049681 -0.006498 -0.108119   
1 -0.050262 -1.315465  1.148418  ... -0.043920 -0.034903  0.033598  0.007487   
2  0.144750 -1.136001  0.788684  ... -0.060885  0.072395  0.168556 -0.022219   
3  0.060882 -0.835030  0.805516  ... -0.032643  0.097366  0.055132  0.032381   
4 -0.097598 -1.196965  0.557457  ... -0.005331  0.056865 -0.009286  0.032854   

        508       509       510       511  \
0  0.106202 -0.070928 -

In [8]:
# ==========================
# 0️⃣ Imports
# ==========================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, Multiply, Add
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ==========================
# 1️⃣ Load MRI features
# ==========================
df_mri = pd.read_csv("mri_features_512d.csv")  # slice-level MRI features
print(df_mri.head())

# Columns 0-511 are numeric MRI features
feature_cols = [str(i) for i in range(512)]

# Ensure numeric (just in case)
df_mri[feature_cols] = df_mri[feature_cols].apply(pd.to_numeric, errors='coerce')
df_mri = df_mri.dropna(subset=feature_cols)

# Extract subject ID from filename
# Example: '002_S_0619_I118678_AD_slice_0079.png' -> '002_S_0619_I118678'
df_mri['subject'] = df_mri['filename'].apply(lambda x: "_".join(x.split("_")[:4]))

# Aggregate MRI features per subject (mean of slices)
df_mri_agg = df_mri.groupby('subject')[feature_cols].mean().reset_index()

# Take first label per subject
df_labels = df_mri.groupby('subject')['label'].first().reset_index()

print("Number of subjects:", df_mri_agg.shape[0])

# ==========================
# 2️⃣ Load BERT embeddings
# ==========================
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")
bert_subject_col = df_text.columns[0]  # first column is subject_id
bert_feature_cols = df_text.columns[1:]  # remaining are numeric features

# Ensure numeric
df_text[bert_feature_cols] = df_text[bert_feature_cols].apply(pd.to_numeric, errors='coerce')
df_text = df_text.dropna(subset=bert_feature_cols)

# ==========================
# 3️⃣ Merge MRI and BERT by subject
# ==========================
df_merged = pd.merge(df_mri_agg, df_text[[bert_subject_col] + list(bert_feature_cols)],
                     left_on='subject', right_on=bert_subject_col, how='inner')

df_labels = pd.merge(df_labels, df_text[[bert_subject_col]],
                     left_on='subject', right_on=bert_subject_col, how='inner')

# Features and labels
X_mri = df_merged[feature_cols].values
X_text = df_merged[bert_feature_cols].values
y_raw = df_labels['label'].values

# Encode labels to 0,1,2
le = LabelEncoder()
y = le.fit_transform(y_raw)
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Final shapes - MRI:", X_mri.shape, "Text:", X_text.shape, "Labels:", y.shape)

# ==========================
# 4️⃣ Scale features
# ==========================
scaler_mri = StandardScaler()
X_mri_scaled = scaler_mri.fit_transform(X_mri)

scaler_text = StandardScaler()
X_text_scaled = scaler_text.fit_transform(X_text)

# ==========================
# 5️⃣ Attention-based fusion model
# ==========================
input_mri = Input(shape=(X_mri_scaled.shape[1],), name="MRI_Input")
input_text = Input(shape=(X_text_scaled.shape[1],), name="Text_Input")

# MRI branch
x1 = Dense(256, activation='relu')(input_mri)
x1 = Dropout(0.3)(x1)

# Text branch
x2 = Dense(256, activation='relu')(input_text)
x2 = Dropout(0.3)(x2)

# Attention layers
att_mri = Dense(1, activation='sigmoid', name='att_mri')(x1)
att_text = Dense(1, activation='sigmoid', name='att_text')(x2)

x1_att = Multiply()([x1, att_mri])
x2_att = Multiply()([x2, att_text])

x = Add()([x1_att, x2_att])
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(len(le.classes_), activation='softmax')(x)

model = Model(inputs=[input_mri, input_text], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==========================
# 6️⃣ Train model
# ==========================
early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    [X_mri_scaled, X_text_scaled],
    y,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

# ==========================
# 7️⃣ Evaluate
# ==========================
loss, acc = model.evaluate([X_mri_scaled, X_text_scaled], y)
print(f"Fused model accuracy: {acc*100:.2f}%")

# ==========================
# 8️⃣ Attention visualization
# ==========================
attention_model = Model(inputs=model.inputs, outputs=[att_mri, att_text])
att_mri_values, att_text_values = attention_model.predict([X_mri_scaled, X_text_scaled])
att_mri_values = att_mri_values.flatten()
att_text_values = att_text_values.flatten()

plt.figure(figsize=(10,5))
plt.scatter(range(len(att_mri_values)), att_mri_values, label='MRI Attention', alpha=0.7)
plt.scatter(range(len(att_text_values)), att_text_values, label='Text Attention', alpha=0.7)
plt.xlabel('Subject index')
plt.ylabel('Attention weight')
plt.title('MRI vs Text Attention Weights per Subject')
plt.legend()
plt.show()

# Average attention per class
for i, class_name in enumerate(le.classes_):
    mask = (y == i)
    print(f"{class_name} - Avg MRI attention: {np.mean(att_mri_values[mask]):.3f}, Avg Text attention: {np.mean(att_text_values[mask]):.3f}")


          0         1         2         3         4         5         6  \
0  3.933728 -6.790685  0.958309 -1.087116 -1.830195 -1.364699  1.263548   
1  4.155911 -8.424635  2.562537 -1.077359 -1.533579 -1.728293  1.085525   
2  2.542780 -6.620580  0.592079 -0.605384 -0.372335 -1.212788  0.575913   
3  1.441322 -5.354963 -0.825883 -0.711870  0.313435 -0.904823  0.239762   
4  2.082709 -6.378020  0.336107 -0.308100 -0.059328 -1.104075  0.314395   

          7         8         9  ...       504       505       506       507  \
0  0.660527 -1.431475  1.493086  ... -0.008533  0.049681 -0.006498 -0.108119   
1 -0.050262 -1.315465  1.148418  ... -0.043920 -0.034903  0.033598  0.007487   
2  0.144750 -1.136001  0.788684  ... -0.060885  0.072395  0.168556 -0.022219   
3  0.060882 -0.835030  0.805516  ... -0.032643  0.097366  0.055132  0.032381   
4 -0.097598 -1.196965  0.557457  ... -0.005331  0.056865 -0.009286  0.032854   

        508       509       510       511  \
0  0.106202 -0.070928 -

ValueError: Found array with 0 sample(s) (shape=(0, 512)) while a minimum of 1 is required by StandardScaler.

In [9]:
print("MRI subjects:", df_mri_agg['subject'].tolist()[:10])
print("BERT subjects:", df_text.iloc[:,0].tolist()[:10])

# Check common subjects
common = set(df_mri_agg['subject']).intersection(set(df_text.iloc[:,0]))
print("Number of common subjects:", len(common))


MRI subjects: ['002_S_0295_I118671', '002_S_0295_I118692', '002_S_0295_I40966', '002_S_0295_I45108', '002_S_0295_I64025', '002_S_0413_I118673', '002_S_0413_I118695', '002_S_0413_I45117', '002_S_0413_I60008', '002_S_0413_I79122']
BERT subjects: []
Number of common subjects: 0


In [10]:
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")
print(df_text.head())
print(df_text.columns)


  Image Data ID   RID        PTID Group    bert_0    bert_1    bert_2  \
0       I112538  1311  941_S_1311   MCI  0.046520 -0.004498 -0.006471   
1        I97341  1311  941_S_1311   MCI  0.048760 -0.003118 -0.015076   
2        I75150  1202  941_S_1202    CN  0.047486 -0.001996 -0.000542   
3       I105437  1202  941_S_1202    CN  0.041326 -0.001045 -0.000135   
4       I108336  1197  941_S_1197    CN  0.050408 -0.005467 -0.003023   

     bert_3    bert_4    bert_5  ...  bert_502  bert_503  bert_504  bert_505  \
0  0.088032 -0.002303  0.052479  ... -0.063068  0.079297 -0.069202 -0.054640   
1  0.088674 -0.002863  0.054960  ... -0.063133  0.081417 -0.064229 -0.055444   
2  0.091288 -0.003527  0.051281  ... -0.068264  0.072590 -0.066312 -0.060175   
3  0.094208 -0.003518  0.050177  ... -0.066339  0.071323 -0.070200 -0.061857   
4  0.090593 -0.003267  0.053620  ... -0.068517  0.071746 -0.069509 -0.056355   

   bert_506  bert_507  bert_508  bert_509  bert_510  bert_511  
0  0.025073 -0.0

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ==========================
# 1️⃣ Load MRI features
# ==========================
df_mri = pd.read_csv("mri_features_512d.csv")

# Extract subject ID from filename (assumes format '..._I<subject>_...')
df_mri['subject'] = df_mri['filename'].apply(lambda x: '_'.join(x.split('_')[0:3]))

# Columns containing features
feature_cols = [col for col in df_mri.columns if col.startswith(tuple(str(i) for i in range(512)))]
label_col = 'label'

# Aggregate MRI slices per subject (mean of slices)
df_mri_agg = df_mri.groupby('subject')[feature_cols].mean().reset_index()
df_labels = df_mri.groupby('subject')[label_col].first().reset_index()

print("MRI aggregated shape:", df_mri_agg.shape)

# ==========================
# 2️⃣ Load BERT embeddings
# ==========================
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")

# Keep only PTID and BERT features
bert_features = [col for col in df_text.columns if col.startswith("bert_")]
df_text_reduced = df_text[['PTID'] + bert_features].copy()
df_text_reduced = df_text_reduced.groupby('PTID')[bert_features].mean().reset_index()  # average if multiple rows per subject

print("BERT embeddings shape after aggregation:", df_text_reduced.shape)

# ==========================
# 3️⃣ Keep only common subjects
# ==========================
common_subjects = set(df_mri_agg['subject']).intersection(set(df_text_reduced['PTID']))
print("Number of common subjects:", len(common_subjects))

df_mri_agg = df_mri_agg[df_mri_agg['subject'].isin(common_subjects)].reset_index(drop=True)
df_labels = df_labels[df_labels['subject'].isin(common_subjects)].reset_index(drop=True)
df_text_reduced = df_text_reduced[df_text_reduced['PTID'].isin(common_subjects)].reset_index(drop=True)

# ==========================
# 4️⃣ Merge MRI + BERT features
# ==========================
df_merged = pd.merge(df_mri_agg, df_text_reduced, left_on='subject', right_on='PTID', how='inner')
X_mri = df_merged[feature_cols].values
X_text = df_merged[bert_features].values

# Labels
y = df_labels['label'].values

# ==========================
# 5️⃣ Encode labels
# ==========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ==========================
# 6️⃣ Scale features
# ==========================
scaler_mri = StandardScaler()
X_mri_scaled = scaler_mri.fit_transform(X_mri)

scaler_text = StandardScaler()
X_text_scaled = scaler_text.fit_transform(X_text)

# ==========================
# 7️⃣ Concatenate features
# ==========================
X_fused = np.concatenate([X_mri_scaled, X_text_scaled], axis=1)
print("Fused features shape:", X_fused.shape)
print("Labels shape:", y_encoded.shape)


MRI aggregated shape: (637, 513)
BERT embeddings shape after aggregation: (639, 513)
Number of common subjects: 637
Classes mapping: {'AD': 0, 'CN': 1, 'MCI': 2}
Fused features shape: (637, 1024)
Labels shape: (637,)


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# ==========================
# 1️⃣ Prepare data
# ==========================
# X_fused: (637, 1024)
# y_encoded: (637,)
# Convert to float32
X = X_fused.astype(np.float32)
y = y_encoded.astype(np.int64)

# Reshape for LSTM: (samples, seq_len, features)
# Here seq_len = 1 (since we have per-subject features), features = 1024
X_lstm = X[:, np.newaxis, :]  # shape -> (637, 1, 1024)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to PyTorch tensors
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================
# 2️⃣ Define LSTM model
# ==========================
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (hn, cn) = self.lstm(x)  # hn: (num_layers, batch, hidden_size)
        out = hn[-1]  # take last layer hidden state
        out = self.fc(out)
        return out

input_size = X_lstm.shape[2]  # 1024
hidden_size = 128
num_layers = 1
num_classes = len(np.unique(y))

model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes)

# ==========================
# 3️⃣ Training setup
# ==========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ==========================
# 4️⃣ Training loop
# ==========================
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()
    
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Evaluate on test set
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total_test += y_batch.size(0)
            correct_test += (predicted == y_batch).sum().item()
    
    test_acc = correct_test / total_test
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

# ==========================
# 5️⃣ Save model
# ==========================
torch.save(model.state_dict(), "fused_LSTM_model.pth")
print("Model saved!")


d:\Anaconda\envs\torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [1/50] - Train Loss: 1.0872, Train Acc: 0.3851, Test Acc: 0.5312
Epoch [2/50] - Train Loss: 0.8358, Train Acc: 0.7289, Test Acc: 0.5391
Epoch [3/50] - Train Loss: 0.6302, Train Acc: 0.8880, Test Acc: 0.5469
Epoch [4/50] - Train Loss: 0.4302, Train Acc: 0.9705, Test Acc: 0.5312
Epoch [5/50] - Train Loss: 0.2673, Train Acc: 0.9980, Test Acc: 0.5859
Epoch [6/50] - Train Loss: 0.1634, Train Acc: 0.9980, Test Acc: 0.6016
Epoch [7/50] - Train Loss: 0.1045, Train Acc: 1.0000, Test Acc: 0.5859
Epoch [8/50] - Train Loss: 0.0716, Train Acc: 1.0000, Test Acc: 0.5703
Epoch [9/50] - Train Loss: 0.0525, Train Acc: 1.0000, Test Acc: 0.5703
Epoch [10/50] - Train Loss: 0.0404, Train Acc: 1.0000, Test Acc: 0.5703
Epoch [11/50] - Train Loss: 0.0322, Train Acc: 1.0000, Test Acc: 0.5859
Epoch [12/50] - Train Loss: 0.0266, Train Acc: 1.0000, Test Acc: 0.5859
Epoch [13/50] - Train Loss: 0.0223, Train Acc: 1.0000, Test Acc: 0.5859
Epoch [14/50] - Train Loss: 0.0191, Train Acc: 1.0000, Test Acc: 0.5859
E

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ==========================
# 1️⃣ Load your data
# ==========================
df_mri = pd.read_csv("mri_features_512d.csv")
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")

# Map labels to 0,1,2
le = LabelEncoder()
df_mri['label_encoded'] = le.fit_transform(df_mri['label'])
classes_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Classes mapping:", classes_mapping)

# Aggregate BERT embeddings per subject (mean)
bert_features_cols = [c for c in df_text.columns if c.startswith("bert_")]
df_text_agg = df_text.groupby("PTID")[bert_features_cols].mean().reset_index()

# Merge MRI and BERT per slice
df_merged = df_mri.merge(df_text_agg, left_on='PTID', right_on='PTID', how='inner')

# ==========================
# 2️⃣ Create sequences per subject
# ==========================
subjects = df_merged['PTID'].unique()
X_seq, y_seq, seq_lengths = [], [], []

for sub in subjects:
    df_sub = df_merged[df_merged['PTID'] == sub].sort_values('filename')
    mri_features = df_sub.iloc[:, :512].values  # MRI features
    bert_features = df_sub[bert_features_cols].values  # BERT features
    fused = np.concatenate([mri_features, bert_features], axis=1)
    X_seq.append(torch.tensor(fused, dtype=torch.float))
    y_seq.append(torch.tensor(df_sub['label_encoded'].iloc[0], dtype=torch.long))
    seq_lengths.append(fused.shape[0])

# Pad sequences
X_padded = pad_sequence(X_seq, batch_first=True)  # shape: (num_subjects, max_seq_len, 1024)
y_tensor = torch.stack(y_seq)
seq_lengths = torch.tensor(seq_lengths)

print("Padded sequences shape:", X_padded.shape)

# ==========================
# 3️⃣ Create Dataset & DataLoader
# ==========================
class MRIDataset(Dataset):
    def __init__(self, X, y, lengths):
        self.X = X
        self.y = y
        self.lengths = lengths
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.lengths[idx]

dataset = MRIDataset(X_padded, y_tensor, seq_lengths)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# ==========================
# 4️⃣ LSTM Model
# ==========================
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=128, num_layers=2, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)  # bidirectional
    def forward(self, x, lengths):
        # Pack padded sequence
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (hn, cn) = self.lstm(packed)
        # Use last hidden state (bidirectional)
        out = torch.cat((hn[-2], hn[-1]), dim=1)
        out = self.fc(out)
        return out

device = "cuda" if torch.cuda.is_available() else "cpu"
model = LSTMClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ==========================
# 5️⃣ Training loop
# ==========================
for epoch in range(1, 21):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch, lengths_batch in loader:
        X_batch, y_batch, lengths_batch = X_batch.to(device), y_batch.to(device), lengths_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch, lengths_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
    print(f"Epoch [{epoch}/20] - Loss: {total_loss/len(dataset):.4f}, Acc: {correct/len(dataset):.4f}")


Classes mapping: {'AD': 0, 'CN': 1, 'MCI': 2}


KeyError: 'PTID'

In [1]:
df_mri['subject'] = df_mri['filename'].apply(
    lambda x: "_".join(x.split("_")[:3])   # 002_S_0619_I118678
)


NameError: name 'df_mri' is not defined

In [2]:
import pandas as pd

df_mri = pd.read_csv("mri_features_512d.csv")


In [3]:
print(df_mri.columns)
print(df_mri.head())


Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '504', '505', '506', '507', '508', '509', '510', '511', 'filename',
       'label'],
      dtype='object', length=514)
          0         1         2         3         4         5         6  \
0  3.933728 -6.790685  0.958309 -1.087116 -1.830195 -1.364699  1.263548   
1  4.155911 -8.424635  2.562537 -1.077359 -1.533579 -1.728293  1.085525   
2  2.542780 -6.620580  0.592079 -0.605384 -0.372335 -1.212788  0.575913   
3  1.441322 -5.354963 -0.825883 -0.711870  0.313435 -0.904823  0.239762   
4  2.082709 -6.378020  0.336107 -0.308100 -0.059328 -1.104075  0.314395   

          7         8         9  ...       504       505       506       507  \
0  0.660527 -1.431475  1.493086  ... -0.008533  0.049681 -0.006498 -0.108119   
1 -0.050262 -1.315465  1.148418  ... -0.043920 -0.034903  0.033598  0.007487   
2  0.144750 -1.136001  0.788684  ... -0.060885  0.072395  0.168556 -0.022219   
3  0.060882 -0.835030  0.805516  ..

In [4]:
df_mri['subject'] = df_mri['filename'].apply(
    lambda x: "_".join(x.split("_")[:3])
)


In [5]:
df_mri[['filename', 'subject']].head()


,filename,subject
0,002_S_0619_I118678_AD_slice_0079.png,002_S_0619
1,002_S_0619_I118678_AD_slice_0080.png,002_S_0619
2,002_S_0619_I118678_AD_slice_0081.png,002_S_0619
3,002_S_0619_I118678_AD_slice_0082.png,002_S_0619
4,002_S_0619_I118678_AD_slice_0083.png,002_S_0619


In [6]:
mri_feature_cols = [c for c in df_mri.columns if c.isdigit()]

df_mri_agg = (
    df_mri
    .groupby(['subject', 'label'])[mri_feature_cols]
    .mean()
    .reset_index()
)


In [7]:
print(df_mri_agg.shape)
df_mri_agg.head()


(637, 514)


,subject,label,0,1,2,3,4,5,6,7,...,502,503,504,505,506,507,508,509,510,511
0,002_S_0295,CN,4.648884,3.419366,-0.042012,0.181123,0.175015,0.281104,0.744090,0.441581,...,-0.008852,0.005427,0.022203,-0.011332,0.014552,0.007542,0.003528,0.015980,-0.005444,0.007891
1,002_S_0413,CN,4.586883,3.014808,0.025967,0.176342,-0.109884,0.611635,0.467495,-0.388415,...,-0.009302,-0.005684,-0.008780,0.025138,0.010422,0.009064,-0.020973,0.003905,-0.015020,0.006425
2,002_S_0619,AD,2.414538,-6.027566,0.389402,-0.007141,-0.011180,-0.902446,0.423219,0.524532,...,-0.010525,0.007991,-0.007584,0.033931,-0.027116,-0.003051,0.001985,0.014993,0.017804,-0.006174
3,002_S_0685,CN,4.411967,3.084508,-0.262853,0.191696,0.001916,0.669510,0.181259,-0.025877,...,0.007133,-0.011223,-0.001139,-0.017890,-0.013939,0.018872,-0.024207,-0.023536,-0.010831,0.006935
4,002_S_0729,MCI,-0.819608,0.713231,-0.756853,-1.005984,0.073850,0.755341,0.468456,-0.035988,...,0.039089,-0.001915,0.002389,-0.008166,-0.036104,0.009503,0.016197,0.023066,0.022077,-0.033874


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_mri_agg['label_id'] = le.fit_transform(df_mri_agg['label'])

print(dict(zip(le.classes_, le.transform(le.classes_))))


{'AD': 0, 'CN': 1, 'MCI': 2}


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt


In [10]:
df_mri = pd.read_csv("mri_features_512d.csv")

# Extract subject
df_mri['subject'] = df_mri['filename'].apply(
    lambda x: "_".join(x.split("_")[:3])
)

# Aggregate MRI slices → subject level
mri_cols = [c for c in df_mri.columns if c.isdigit()]

df_mri_agg = (
    df_mri
    .groupby(['subject', 'label'])[mri_cols]
    .mean()
    .reset_index()
)


In [11]:
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")

bert_cols = [c for c in df_text.columns if c.startswith("bert_")]

df_text_agg = (
    df_text
    .groupby("PTID")[bert_cols]
    .mean()
    .reset_index()
)


In [14]:
df_meta = pd.read_csv("ADNI_metadata.csv")

df_meta['subject'] = df_meta['Image Data ID'].str.extract(r'(.*)_I\d+')

df_map = df_meta[['subject', 'PTID']].drop_duplicates()


FileNotFoundError: [Errno 2] No such file or directory: 'ADNI_metadata.csv'

In [15]:
import pandas as pd
import numpy as np

df_mri = pd.read_csv("mri_features_512d.csv")

# Extract Image Data ID (Ixxxxx)
df_mri['Image Data ID'] = df_mri['filename'].str.extract(r'(I\d+)')

df_mri.head()


,0,1,2,3,4,5,6,7,8,9,...,505,506,507,508,509,510,511,filename,label,Image Data ID
0,3.933728,-6.790685,0.958309,-1.087116,-1.830195,-1.364699,1.263548,0.660527,-1.431475,1.493086,...,0.049681,-0.006498,-0.108119,0.106202,-0.070928,-0.004630,0.100422,002_S_0619_I118678_AD_slice_0079.png,AD,I118678
1,4.155911,-8.424635,2.562537,-1.077359,-1.533579,-1.728293,1.085525,-0.050262,-1.315465,1.148418,...,-0.034903,0.033598,0.007487,0.148335,-0.021368,-0.002512,0.049608,002_S_0619_I118678_AD_slice_0080.png,AD,I118678
2,2.542780,-6.620580,0.592079,-0.605384,-0.372335,-1.212788,0.575913,0.144750,-1.136001,0.788684,...,0.072395,0.168556,-0.022219,0.166609,-0.014115,0.007277,0.046518,002_S_0619_I118678_AD_slice_0081.png,AD,I118678
3,1.441322,-5.354963,-0.825883,-0.711870,0.313435,-0.904823,0.239762,0.060882,-0.835030,0.805516,...,0.097366,0.055132,0.032381,0.122927,-0.026041,0.007733,0.006380,002_S_0619_I118678_AD_slice_0082.png,AD,I118678
4,2.082709,-6.378020,0.336107,-0.308100,-0.059328,-1.104075,0.314395,-0.097598,-1.196965,0.557457,...,0.056865,-0.009286,0.032854,0.081114,-0.139952,0.114629,0.010963,002_S_0619_I118678_AD_slice_0083.png,AD,I118678


In [16]:
mri_cols = [c for c in df_mri.columns if c.isdigit()]

df_mri_img = (
    df_mri
    .groupby(['Image Data ID', 'label'])[mri_cols]
    .mean()
    .reset_index()
)


In [17]:
df_text = pd.read_csv("ADNI_BERT_embeddings_512.csv")

bert_cols = [c for c in df_text.columns if c.startswith("bert_")]

df_text.head()


,Image Data ID,RID,PTID,Group,bert_0,bert_1,bert_2,bert_3,bert_4,bert_5,...,bert_502,bert_503,bert_504,bert_505,bert_506,bert_507,bert_508,bert_509,bert_510,bert_511
0,I112538,1311,941_S_1311,MCI,0.046520,-0.004498,-0.006471,0.088032,-0.002303,0.052479,...,-0.063068,0.079297,-0.069202,-0.054640,0.025073,-0.030114,0.029292,0.062412,-0.007135,0.075038
1,I97341,1311,941_S_1311,MCI,0.048760,-0.003118,-0.015076,0.088674,-0.002863,0.054960,...,-0.063133,0.081417,-0.064229,-0.055444,0.024335,-0.029304,0.035903,0.061687,-0.007822,0.072469
2,I75150,1202,941_S_1202,CN,0.047486,-0.001996,-0.000542,0.091288,-0.003527,0.051281,...,-0.068264,0.072590,-0.066312,-0.060175,0.016715,-0.031600,0.034395,0.071990,-0.010264,0.067010
3,I105437,1202,941_S_1202,CN,0.041326,-0.001045,-0.000135,0.094208,-0.003518,0.050177,...,-0.066339,0.071323,-0.070200,-0.061857,0.013668,-0.033093,0.030954,0.072318,-0.012171,0.069062
4,I108336,1197,941_S_1197,CN,0.050408,-0.005467,-0.003023,0.090593,-0.003267,0.053620,...,-0.068517,0.071746,-0.069509,-0.056355,0.021591,-0.032837,0.032774,0.065714,-0.009255,0.072653


In [18]:
df_text_img = (
    df_text
    .groupby(['Image Data ID', 'Group'])[bert_cols]
    .mean()
    .reset_index()
)


In [19]:
df_fused = df_mri_img.merge(
    df_text_img,
    on='Image Data ID',
    how='inner'
)

print("Fused shape:", df_fused.shape)


Fused shape: (1435, 1027)


In [20]:
X_mri = df_fused[mri_cols].values
X_text = df_fused[bert_cols].values

X = np.hstack([X_mri, X_text])

y = df_fused['label'].map({'AD': 0, 'CN': 1, 'MCI': 2}).values


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [23]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [24]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['AD','CN','MCI']))


Accuracy: 0.975609756097561
              precision    recall  f1-score   support

          AD       0.98      0.98      0.98        61
          CN       0.99      0.94      0.96        84
         MCI       0.97      0.99      0.98       142

    accuracy                           0.98       287
   macro avg       0.98      0.97      0.98       287
weighted avg       0.98      0.98      0.98       287



In [25]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# --------------------------------------------------
# Train-test split (same split for fairness)
# --------------------------------------------------
X_train_mri, X_test_mri, y_train, y_test = train_test_split(
    X_mri, y, test_size=0.2, stratify=y, random_state=42
)

X_train_bert, X_test_bert, _, _ = train_test_split(
    X_bert, y, test_size=0.2, stratify=y, random_state=42
)

X_train_fused = np.concatenate([X_train_mri, X_train_bert], axis=1)
X_test_fused  = np.concatenate([X_test_mri, X_test_bert], axis=1)

# --------------------------------------------------
# XGBoost config
# --------------------------------------------------
def train_xgb(Xtr, ytr, Xte):
    model = xgb.XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42
    )
    model.fit(Xtr, ytr)
    return accuracy_score(y_test, model.predict(Xte)), model

# --------------------------------------------------
# Train models
# --------------------------------------------------
acc_mri, model_mri = train_xgb(X_train_mri, y_train, X_test_mri)
acc_bert, model_bert = train_xgb(X_train_bert, y_train, X_test_bert)
acc_fused, model_fused = train_xgb(X_train_fused, y_train, X_test_fused)

print(f"MRI only accuracy  : {acc_mri:.4f}")
print(f"BERT only accuracy : {acc_bert:.4f}")
print(f"Fused accuracy     : {acc_fused:.4f}")
print(f"BERT contribution  : +{acc_fused - acc_mri:.4f}")


NameError: name 'X_bert' is not defined

In [26]:
# MRI features
X_mri = df_mri.iloc[:, :-2].values   # remove filename + label
y = df_mri['label'].map({'AD': 0, 'CN': 1, 'MCI': 2}).values


In [27]:
# BERT features
X_bert = df_bert.iloc[:, :-1].values   # remove subject column


NameError: name 'df_bert' is not defined

In [28]:
X_bert = df_text_agg.filter(like="bert_").values


In [29]:
X_bert = df_text_agg.iloc[:, 1:].values   # drop subject column


In [30]:
print(X_mri.shape)
print(X_bert.shape)
print(y.shape)


(157312, 513)
(639, 512)
(157312,)
